# Constrained metabolism model

This notebook builds and solves the constrained metabolism model described in
Supplementary Methods 5. It selects one of 19 manifest-registered scenarios,
validates its inputs, and writes scenario-specific results under `results/`.

All categorical values use the Supplementary Information codes. Python identifiers
and table fields use descriptive `snake_case`. Each flow record is assigned a unit
carbon-impact coefficient in kg CO2-eq per tonne.

Definitions of node types, materials, parameters, recycling schemes, and constraint
groups are provided in [`data/codebook.csv`](../data/codebook.csv). Input and output
schemas are documented in [`data/DATA_SCHEMA.md`](../data/DATA_SCHEMA.md).

## 1. Configuration and input validation

In [ ]:
# ============================================================
# Repository configuration
# ============================================================
from pathlib import Path
import csv
import hashlib
import re
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp
import gurobipy as gp
from gurobipy import GRB

# Select one of the 19 scenario IDs registered in data/scenario_manifest.csv.
SCENARIO_ID = "l_n_p50"

# Runtime and reporting settings (user-adjustable).
TIME_LIMIT_SECONDS = 3600
REPORTING_THRESHOLD_T_PER_YEAR = 1e-6


def log_status(message):
    """Print one timestamped model-construction message."""
    print(f'{time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())} - {message}')

WORKING_DIRECTORY = Path.cwd().resolve()
if (WORKING_DIRECTORY / "data" / "scenario_manifest.csv").is_file():
    REPOSITORY_ROOT = WORKING_DIRECTORY
elif (WORKING_DIRECTORY.parent / "data" / "scenario_manifest.csv").is_file():
    REPOSITORY_ROOT = WORKING_DIRECTORY.parent
else:
    raise FileNotFoundError(
        "Repository data directory not found. Start Jupyter from the repository "
        "root or from its notebooks directory."
    )

DATA_DIRECTORY = REPOSITORY_ROOT / "data"

SCENARIO_MANIFEST_PATH = DATA_DIRECTORY / "scenario_manifest.csv"
FILE_MANIFEST_PATH = DATA_DIRECTORY / "file_manifest.csv"
TRANSPORT_NETWORK_RELATIVE_PATHS = (
    "network/facility_project_arcs_km.csv",
    "network/grid_to_recycling_arcs_km.csv",
)
NODE_PARAMETERS_RELATIVE_PATH = "model/node_parameters_2026.csv"

SCENARIO_MANIFEST_COLUMNS = [
    "scenario_id",
    "scenario_type",
    "lifespan_scenario",
    "policy_scenario",
    "input_realization",
    "waste_concrete_file",
    "waste_brick_file",
    "waste_brick_absorption_rate_pct",
]
FILE_MANIFEST_COLUMNS = [
    "release_path",
    "file_role",
    "scenario_id",
    "material_code",
    "row_count",
    "column_names",
    "size_bytes",
    "sha256",
]

NODE_TYPES = {
    "NDPRJ", "NDCBP", "NDBMP", "NDCEM", "NDSND",
    "NDQRY", "NDWRP", "NDGRD", "NDCWP", "NDCWV",
}
NODE_ID_PATTERN = re.compile(
    r"^(NDPRJ|NDCBP|NDBMP|NDCEM|NDSND|NDQRY|NDWRP|NDGRD|NDCWP|NDCWV)_([1-9][0-9]*)$"
)
ALLOWED_ARC_TYPES = {
    "NDCEM->NDCWV", "NDSND->NDCWV", "NDQRY->NDCWV",
    "NDCWV->NDCWP", "NDCWP->NDCBP", "NDCWP->NDBMP",
    "NDCBP->NDPRJ", "NDBMP->NDPRJ", "NDGRD->NDWRP",
    "NDWRP->NDPRJ", "NDWRP->NDCBP", "NDWRP->NDBMP",
}
STATIC_PARAMETER_BY_NODE_TYPE = {
    "NDPRJ": {"concrete_demand", "block_demand"},
    "NDCBP": {"batching_plant_capacity"},
    "NDBMP": {"block_plant_capacity"},
    "NDCEM": {"cement_supply_capacity"},
    "NDSND": {"marine_sand_supply_capacity"},
    "NDQRY": {"natural_coarse_aggregate_supply_capacity"},
    "NDWRP": {"recycling_plant_capacity"},
}


def read_csv_records(path, expected_columns):
    """Read a small CSV and require its exact schema."""
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != expected_columns:
            raise ValueError(
                f"Unexpected columns in {path.name}: {reader.fieldnames}; "
                f"expected {expected_columns}."
            )
        return list(reader)


def resolve_data_path(relative_path):
    """Resolve a data-directory-relative path and reject path traversal."""
    if not relative_path or Path(relative_path).is_absolute():
        raise ValueError(f"Data path must be non-empty and repository-relative: {relative_path!r}")
    resolved_path = (DATA_DIRECTORY / relative_path).resolve()
    try:
        resolved_path.relative_to(DATA_DIRECTORY.resolve())
    except ValueError as error:
        raise ValueError(f"Data path points outside the data directory: {relative_path!r}") from error
    if not resolved_path.is_file():
        raise FileNotFoundError(f"Required data file not found: {relative_path}")
    return resolved_path


def resolve_manifest_release_path(release_path):
    """Resolve a repository-relative manifest path confined to data/."""
    candidate = Path(release_path)
    if not release_path or candidate.is_absolute() or not candidate.parts:
        raise ValueError(
            f"Manifest release_path must be repository-relative: {release_path!r}"
        )
    if candidate.parts[0] != "data":
        raise ValueError(
            f"Manifest release_path must start with 'data/': {release_path!r}"
        )
    resolved_path = (REPOSITORY_ROOT / candidate).resolve()
    try:
        resolved_path.relative_to(DATA_DIRECTORY.resolve())
    except ValueError as error:
        raise ValueError(
            f"Manifest release_path points outside data/: {release_path!r}"
        ) from error
    if not resolved_path.is_file():
        raise FileNotFoundError(f"Manifest-listed file not found: {release_path}")
    return resolved_path


def sha256_file(path):
    """Return the SHA-256 digest of one file."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def inspect_csv_file(path):
    """Return a CSV header and data-row count without loading the table."""
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        try:
            header = next(reader)
        except StopIteration as error:
            raise ValueError(f"CSV file is empty: {path}") from error
        row_count = sum(1 for _ in reader)
    return header, row_count


scenario_records = read_csv_records(SCENARIO_MANIFEST_PATH, SCENARIO_MANIFEST_COLUMNS)
if len(scenario_records) != 19:
    raise ValueError(f"Scenario manifest must contain 19 records; found {len(scenario_records)}.")

scenario_ids = [record["scenario_id"] for record in scenario_records]
if len(set(scenario_ids)) != len(scenario_ids):
    raise ValueError("Scenario IDs must be unique.")

expected_scenario_ids = {"baseline"}
for lifespan_code in ("s", "m", "l"):
    for policy_code in ("n", "ml"):
        for realization_code in ("p2.5", "p50", "p97.5"):
            expected_scenario_ids.add(f"{lifespan_code}_{policy_code}_{realization_code}")
if set(scenario_ids) != expected_scenario_ids:
    raise ValueError("Scenario manifest does not contain the expected compact scenario-ID set.")

lifespan_by_code = {"s": "short", "m": "medium", "l": "long"}
policy_by_code = {"n": "near", "ml": "mid-long"}
realization_by_code = {"p2.5": "P2.5", "p50": "P50", "p97.5": "P97.5"}

for record in scenario_records:
    scenario_id = record["scenario_id"]
    if scenario_id == "baseline":
        expected_baseline_record = {
            "scenario_id": "baseline",
            "scenario_type": "baseline",
            "lifespan_scenario": "",
            "policy_scenario": "",
            "input_realization": "",
            "waste_concrete_file": "baseline/baseline_wc.csv",
            "waste_brick_file": "baseline/baseline_wb.csv",
            "waste_brick_absorption_rate_pct": "",
        }
        if record != expected_baseline_record:
            raise ValueError("The baseline scenario record does not match the expected manifest values.")
    else:
        if record["scenario_type"] != "integrated":
            raise ValueError(f"Integrated scenario has an invalid scenario_type: {scenario_id}")
        lifespan_code, policy_code, realization_code = scenario_id.split("_", 2)
        expected_metadata = (
            lifespan_by_code[lifespan_code],
            policy_by_code[policy_code],
            realization_by_code[realization_code],
        )
        observed_metadata = (
            record["lifespan_scenario"],
            record["policy_scenario"],
            record["input_realization"],
        )
        if observed_metadata != expected_metadata:
            raise ValueError(f"Scenario metadata do not match compact ID {scenario_id!r}.")
        try:
            absorption_rate = float(record["waste_brick_absorption_rate_pct"])
        except ValueError as error:
            raise ValueError(f"Invalid waste-brick absorption rate for {scenario_id!r}.") from error
        if not np.isfinite(absorption_rate) or not (0 < absorption_rate <= 100):
            raise ValueError(f"Waste-brick absorption rate is outside (0, 100] for {scenario_id!r}.")

    resolve_data_path(record["waste_concrete_file"])
    resolve_data_path(record["waste_brick_file"])

waste_input_paths = [
    record[field]
    for record in scenario_records
    for field in ("waste_concrete_file", "waste_brick_file")
]
if len(set(waste_input_paths)) != 38:
    raise ValueError("The scenario manifest must reference 38 distinct WC/WB input files.")

selected_records = [record for record in scenario_records if record["scenario_id"] == SCENARIO_ID]
if len(selected_records) != 1:
    raise ValueError(f"SCENARIO_ID must resolve to exactly one manifest record: {SCENARIO_ID!r}")
selected_scenario = selected_records[0]

RESULTS_ROOT = (REPOSITORY_ROOT / "results").resolve()
RESULTS_DIRECTORY = (RESULTS_ROOT / SCENARIO_ID).resolve()
try:
    RESULTS_DIRECTORY.relative_to(RESULTS_ROOT)
except ValueError as error:
    raise ValueError("Scenario results path points outside results/.") from error
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

TRANSPORT_NETWORK_PATHS = tuple(
    resolve_data_path(relative_path)
    for relative_path in TRANSPORT_NETWORK_RELATIVE_PATHS
)
NODE_PARAMETERS_PATH = resolve_data_path(NODE_PARAMETERS_RELATIVE_PATH)
WASTE_CONCRETE_INPUT_RELATIVE_PATH = selected_scenario["waste_concrete_file"]
WASTE_BRICK_INPUT_RELATIVE_PATH = selected_scenario["waste_brick_file"]
WASTE_CONCRETE_INPUT_PATH = resolve_data_path(WASTE_CONCRETE_INPUT_RELATIVE_PATH)
WASTE_BRICK_INPUT_PATH = resolve_data_path(WASTE_BRICK_INPUT_RELATIVE_PATH)

FLOW_RESULTS_PATH = RESULTS_DIRECTORY / f"flow_results_{SCENARIO_ID}.csv"
SCHEME_THROUGHPUT_RESULTS_PATH = RESULTS_DIRECTORY / f"scheme_throughput_{SCENARIO_ID}.csv"
INFEASIBILITY_REPORT_PATH = RESULTS_DIRECTORY / f"infeasibility_report_{SCENARIO_ID}.ilp"

file_manifest_records = read_csv_records(FILE_MANIFEST_PATH, FILE_MANIFEST_COLUMNS)
if len(file_manifest_records) != 42:
    raise ValueError(f"File manifest must contain 42 records; found {len(file_manifest_records)}.")
file_manifest_by_path = {record["release_path"]: record for record in file_manifest_records}
if len(file_manifest_by_path) != len(file_manifest_records):
    raise ValueError("File-manifest release paths must be unique.")

expected_file_manifest_paths = {
    "data/scenario_manifest.csv",
    *(f"data/{relative_path}" for relative_path in TRANSPORT_NETWORK_RELATIVE_PATHS),
    f"data/{NODE_PARAMETERS_RELATIVE_PATH}",
    *(f"data/{relative_path}" for relative_path in waste_input_paths),
}
if set(file_manifest_by_path) != expected_file_manifest_paths:
    raise ValueError("File manifest contains missing or unknown release paths.")

expected_fixed_manifest_metadata = {
    "data/scenario_manifest.csv": ("scenario_manifest", "", ""),
    **{
        f"data/{relative_path}": ("network_input", "", "")
        for relative_path in TRANSPORT_NETWORK_RELATIVE_PATHS
    },
    f"data/{NODE_PARAMETERS_RELATIVE_PATH}": ("model_input", "", ""),
}
for release_path, expected_metadata in expected_fixed_manifest_metadata.items():
    record = file_manifest_by_path[release_path]
    observed_metadata = (
        record["file_role"], record["scenario_id"], record["material_code"]
    )
    if observed_metadata != expected_metadata:
        raise ValueError(f"Invalid file-manifest metadata for {release_path!r}.")

for scenario_record in scenario_records:
    expected_role = (
        "baseline_input"
        if scenario_record["scenario_type"] == "baseline"
        else "scenario_input"
    )
    for material_code, field_name in (
        ("WC", "waste_concrete_file"),
        ("WB", "waste_brick_file"),
    ):
        release_path = f"data/{scenario_record[field_name]}"
        record = file_manifest_by_path[release_path]
        observed_metadata = (
            record["file_role"], record["scenario_id"], record["material_code"]
        )
        expected_metadata = (
            expected_role, scenario_record["scenario_id"], material_code
        )
        if observed_metadata != expected_metadata:
            raise ValueError(f"Invalid file-manifest metadata for {release_path!r}.")

for relative_path, record in file_manifest_by_path.items():
    physical_path = resolve_manifest_release_path(relative_path)
    observed_columns, observed_row_count = inspect_csv_file(physical_path)
    expected_columns = record["column_names"].split("|")
    if observed_columns != expected_columns:
        raise ValueError(f"File-manifest column mismatch for {relative_path!r}.")
    if observed_row_count != int(record["row_count"]):
        raise ValueError(f"File-manifest row-count mismatch for {relative_path!r}.")
    if physical_path.stat().st_size != int(record["size_bytes"]):
        raise ValueError(f"File-manifest byte-count mismatch for {relative_path!r}.")
    if sha256_file(physical_path) != record["sha256"]:
        raise ValueError(f"File-manifest SHA-256 mismatch for {relative_path!r}.")


def require_exact_columns(table, expected_columns, table_name):
    """Require an exact ordered DataFrame schema."""
    observed_columns = table.columns.tolist()
    if observed_columns != expected_columns:
        raise ValueError(
            f"Unexpected columns in {table_name}: {observed_columns}; expected {expected_columns}."
        )


def require_complete_strings(table, column_name, table_name):
    """Require non-missing, non-blank string values."""
    if table[column_name].isna().any():
        raise ValueError(f"Missing {column_name} value in {table_name}.")
    stripped = table[column_name].astype(str).str.strip()
    if stripped.eq("").any():
        raise ValueError(f"Blank {column_name} value in {table_name}.")
    table[column_name] = stripped


def require_nonnegative_finite(table, column_name, table_name):
    """Convert one numeric column to float64 and require finite non-negative values."""
    try:
        values = pd.to_numeric(table[column_name], errors="raise").to_numpy(dtype=np.float64)
    except (TypeError, ValueError) as error:
        raise ValueError(f"Non-numeric {column_name} value in {table_name}.") from error
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f"Non-finite or negative {column_name} value in {table_name}.")
    table[column_name] = values


def canonical_node_type(node_id):
    """Return the node type encoded in a validated node ID."""
    match = NODE_ID_PATTERN.fullmatch(str(node_id))
    if match is None:
        raise ValueError(f"Invalid node ID: {node_id!r}")
    return match.group(1)


transport_network_parts = []
expected_transport_partition_arc_types = (
    ALLOWED_ARC_TYPES - {"NDGRD->NDWRP"},
    {"NDGRD->NDWRP"},
)
expected_transport_partition_row_counts = (2_382_778, 2_814_350)
for relative_path, physical_path, expected_arc_types, expected_row_count in zip(
    TRANSPORT_NETWORK_RELATIVE_PATHS,
    TRANSPORT_NETWORK_PATHS,
    expected_transport_partition_arc_types,
    expected_transport_partition_row_counts,
):
    transport_network_part_df = pd.read_csv(physical_path)
    table_name = f"transport-network partition {relative_path}"
    require_exact_columns(
        transport_network_part_df,
        ["origin_node_id", "destination_node_id", "distance_km"],
        table_name,
    )
    for column_name in ("origin_node_id", "destination_node_id"):
        require_complete_strings(transport_network_part_df, column_name, table_name)
    require_nonnegative_finite(transport_network_part_df, "distance_km", table_name)
    transport_network_part_df["origin_node_type"] = transport_network_part_df["origin_node_id"].map(canonical_node_type)
    transport_network_part_df["destination_node_type"] = transport_network_part_df[
        "destination_node_id"
    ].map(canonical_node_type)
    transport_network_part_df["arc_type"] = (
        transport_network_part_df["origin_node_type"]
        + "->"
        + transport_network_part_df["destination_node_type"]
    )
    observed_partition_arc_types = set(transport_network_part_df["arc_type"])
    if observed_partition_arc_types != expected_arc_types:
        raise ValueError(
            f"Unexpected arc types in {table_name}: {sorted(observed_partition_arc_types)}"
        )
    if len(transport_network_part_df) != expected_row_count:
        raise ValueError(
            f"Unexpected row count in {table_name}: {len(transport_network_part_df)}"
        )
    transport_network_parts.append(transport_network_part_df)

transport_network_df = pd.concat(
    transport_network_parts, ignore_index=True, copy=False
)
del transport_network_parts, transport_network_part_df
if transport_network_df.duplicated(["origin_node_id", "destination_node_id"]).any():
    raise ValueError("Duplicate origin/destination pair in the transport network.")

observed_arc_types = set(transport_network_df["arc_type"])
if observed_arc_types != ALLOWED_ARC_TYPES:
    raise ValueError(
        f"Transport-network arc types differ from the allowed registry: "
        f"{sorted(observed_arc_types)}"
    )
if len(transport_network_df) != 5_197_128:
    raise ValueError(f"Unexpected transport-network row count: {len(transport_network_df)}")

node_parameters_df = pd.read_csv(NODE_PARAMETERS_PATH)
require_exact_columns(
    node_parameters_df,
    ["node_id", "parameter_name", "value_t_per_year"],
    "static node parameters",
)
for column_name in ("node_id", "parameter_name"):
    require_complete_strings(node_parameters_df, column_name, "static node parameters")
require_nonnegative_finite(node_parameters_df, "value_t_per_year", "static node parameters")
node_parameters_df["node_type"] = node_parameters_df["node_id"].map(canonical_node_type)

if node_parameters_df.duplicated(["node_id", "parameter_name"]).any():
    raise ValueError("Duplicate (node_id, parameter_name) key in static node parameters.")

for row in node_parameters_df.itertuples(index=False):
    expected_parameters = STATIC_PARAMETER_BY_NODE_TYPE.get(row.node_type)
    if expected_parameters is None or row.parameter_name not in expected_parameters:
        raise ValueError(
            f"Parameter {row.parameter_name!r} is incompatible with node {row.node_id!r}."
        )

network_node_ids_by_type = {
    node_type: set(
        pd.concat(
            [
                transport_network_df.loc[
                    transport_network_df["origin_node_type"] == node_type, "origin_node_id"
                ],
                transport_network_df.loc[
                    transport_network_df["destination_node_type"] == node_type, "destination_node_id"
                ],
            ],
            ignore_index=True,
        )
    )
    for node_type in NODE_TYPES
}
for node_type, expected_parameters in STATIC_PARAMETER_BY_NODE_TYPE.items():
    node_ids = network_node_ids_by_type[node_type]
    expected_pairs = {
        (node_id, parameter_name)
        for node_id in node_ids
        for parameter_name in expected_parameters
    }
    observed_pairs = set(
        node_parameters_df.loc[
            node_parameters_df["node_type"] == node_type,
            ["node_id", "parameter_name"],
        ].itertuples(index=False, name=None)
    )
    if observed_pairs != expected_pairs:
        raise ValueError(f"Static parameter coverage mismatch for node type {node_type}.")

# WC and WB files contain model-ready annual network inputs. Values reflect
# applicable city-specific policy targets; WB also includes the applicable
# absorption adjustment.
waste_concrete_input_df = pd.read_csv(WASTE_CONCRETE_INPUT_PATH)
require_exact_columns(
    waste_concrete_input_df,
    ["grid_cell_id", "waste_concrete_network_input_t_per_year"],
    "waste-concrete input",
)
waste_brick_input_df = pd.read_csv(WASTE_BRICK_INPUT_PATH)
require_exact_columns(
    waste_brick_input_df,
    ["grid_cell_id", "waste_brick_network_input_t_per_year"],
    "waste-brick input",
)

for table, value_column, table_name in (
    (
        waste_concrete_input_df,
        "waste_concrete_network_input_t_per_year",
        "waste-concrete input",
    ),
    (
        waste_brick_input_df,
        "waste_brick_network_input_t_per_year",
        "waste-brick input",
    ),
):
    require_complete_strings(table, "grid_cell_id", table_name)
    require_nonnegative_finite(table, value_column, table_name)
    if table["grid_cell_id"].duplicated().any():
        raise ValueError(f"Duplicate grid_cell_id in {table_name}.")
    node_types = table["grid_cell_id"].map(canonical_node_type)
    if set(node_types) != {"NDGRD"}:
        raise ValueError(f"Non-grid node ID in {table_name}.")

expected_grid_cell_ids = network_node_ids_by_type["NDGRD"]
if set(waste_concrete_input_df["grid_cell_id"]) != expected_grid_cell_ids:
    raise ValueError("Waste-concrete grid-cell coverage does not match the transport network.")
if set(waste_brick_input_df["grid_cell_id"]) != expected_grid_cell_ids:
    raise ValueError("Waste-brick grid-cell coverage does not match the transport network.")

# DataFrame.pivot is intentionally strict: it raises instead of aggregating duplicate keys.
node_parameter_matrix_df = node_parameters_df.pivot(
    index="node_id",
    columns="parameter_name",
    values="value_t_per_year",
)
waste_concrete_input_by_grid_cell = waste_concrete_input_df.set_index("grid_cell_id")[
    "waste_concrete_network_input_t_per_year"
]
waste_brick_input_by_grid_cell = waste_brick_input_df.set_index("grid_cell_id")[
    "waste_brick_network_input_t_per_year"
]

scheme_throughput_vars = None
scheme_throughput_index_df = None

print(f"Scenario: {SCENARIO_ID}")
print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Results directory: {RESULTS_DIRECTORY}")
print(f"Waste-concrete input: {WASTE_CONCRETE_INPUT_RELATIVE_PATH}")
print(f"Waste-brick input: {WASTE_BRICK_INPUT_RELATIVE_PATH}")

# ============================================================
# Replacement and mix parameters (user-adjustable)
# ============================================================
# Each replacement ratio is substitute / (virgin + substitute).

# C1: direct RCS and RCL use at construction-project nodes.
RCS_MAX_REPLACEMENT_RATIO = 0.50
RCL_MAX_REPLACEMENT_RATIO = 0.30

# C3: RCC replacement in concrete delivered by batching plants.
RCC_MAX_COARSE_AGG_REPLACEMENT_RATIO = 0.50

# C5: replacement in blocks delivered by block manufacturing plants.
RCF_RBF_MAX_FINE_AGG_REPLACEMENT_RATIO = 0.75
RBC_MAX_COARSE_AGG_REPLACEMENT_RATIO = 0.25
RBP_MAX_CEMENT_REPLACEMENT_RATIO = 0.20

# C3 concrete mix: CEM : NS : (NCA + RCC) : WAT.
CONCRETE_CEMENT_MASS_RATIO = 385
CONCRETE_NATURAL_SAND_MASS_RATIO = 714
CONCRETE_COARSE_AGGREGATE_MASS_RATIO = 1071
CONCRETE_WATER_MASS_RATIO = 230

# C5 block mix: (CEM + RBP) : (NS + RCF + RBF) : (NCA + RBC) : WAT.
BLOCK_CEMENT_AND_BRICK_POWDER_MASS_RATIO = 338
BLOCK_FINE_AGGREGATE_MASS_RATIO = 788
BLOCK_COARSE_AGGREGATE_MASS_RATIO = 1044
BLOCK_WATER_MASS_RATIO = 230

## 2. Material routes and carbon-impact coefficients

The following cells expand the 17 material-route templates into arc-level flow
variables, attach network distances, and calculate each flow variable's unit
carbon-impact coefficient. Coefficients are expressed in kg CO2-eq t^-1; their
basis is described in Supplementary Methods 5.

In [ ]:
def build_material_arc_table(
    material_code,
    origin_node_type,
    destination_node_type,
    transport_network_local_df,
):
    """Return network arcs matching one material and allowed arc type."""
    arc_type_filter = f"{origin_node_type}->{destination_node_type}"
    matching_arcs_df = transport_network_local_df.loc[
        transport_network_local_df["arc_type"] == arc_type_filter
    ]
    return pd.DataFrame(
        {
            "material_code": material_code,
            "origin_node_id": matching_arcs_df["origin_node_id"],
            "destination_node_id": matching_arcs_df["destination_node_id"],
        }
    )


material_route_templates = [
    ("CEM", "NDCEM_i -> NDCWV_i -> NDCWP_j -> NDCBP_k -> NDPRJ_m"),
    ("CEM", "NDCEM_i -> NDCWV_i -> NDCWP_j -> NDBMP_k -> NDPRJ_m"),
    ("NS", "NDSND_i -> NDCWV_i -> NDCWP_j -> NDCBP_k -> NDPRJ_m"),
    ("NS", "NDSND_i -> NDCWV_i -> NDCWP_j -> NDBMP_k -> NDPRJ_m"),
    ("NCA", "NDQRY_i -> NDCWV_i -> NDCWP_j -> NDCBP_k -> NDPRJ_m"),
    ("NCA", "NDQRY_i -> NDCWV_i -> NDCWP_j -> NDBMP_k -> NDPRJ_m"),
    ("WAT", "NDCBP_k -> NDPRJ_m"),
    ("WAT", "NDBMP_k -> NDPRJ_m"),
    ("WC", "NDGRD_p -> NDWRP_q"),
    ("WB", "NDGRD_p -> NDWRP_q"),
    ("RCS", "NDWRP_q -> NDPRJ_r"),
    ("RCL", "NDWRP_q -> NDPRJ_r"),
    ("RCC", "NDWRP_q -> NDCBP_k -> NDPRJ_r"),
    ("RCF", "NDWRP_q -> NDBMP_s -> NDPRJ_r"),
    ("RBC", "NDWRP_q -> NDBMP_s -> NDPRJ_r"),
    ("RBF", "NDWRP_q -> NDBMP_s -> NDPRJ_r"),
    ("RBP", "NDWRP_q -> NDBMP_s -> NDPRJ_r"),
]

if len(material_route_templates) != 17:
    raise RuntimeError("The route registry must contain 17 material-route templates.")

material_arc_type_registry = set()
for material_code, route_template in material_route_templates:
    node_types_in_route = [
        point.split("_")[0] for point in route_template.split(" -> ")
    ]
    material_arc_type_registry.update(
        (material_code, f"{origin_node_type}->{destination_node_type}")
        for origin_node_type, destination_node_type in zip(
            node_types_in_route[:-1], node_types_in_route[1:]
        )
    )
if len(material_arc_type_registry) != 34:
    raise RuntimeError("The route registry must contain 34 material/arc-type pairs.")

material_arc_tables = []
for material_code, route_template in material_route_templates:
    node_types_in_route = [
        point.split("_")[0] for point in route_template.split(" -> ")
    ]
    for origin_node_type, destination_node_type in zip(
        node_types_in_route[:-1], node_types_in_route[1:]
    ):
        material_arc_tables.append(
            build_material_arc_table(
                material_code,
                origin_node_type,
                destination_node_type,
                transport_network_df,
            )
        )

flow_variable_index_df = pd.concat(material_arc_tables, ignore_index=True)
flow_variable_index_df.drop_duplicates(
    subset=["material_code", "origin_node_id", "destination_node_id"],
    inplace=True,
)

flow_variable_index_df = flow_variable_index_df.merge(
    transport_network_df[
        [
            "origin_node_id",
            "destination_node_id",
            "distance_km",
            "arc_type",
            "origin_node_type",
            "destination_node_type",
        ]
    ],
    on=["origin_node_id", "destination_node_id"],
    how="left",
    validate="many_to_one",
)
if flow_variable_index_df["distance_km"].isna().any():
    raise ValueError("A material arc has no matching network distance.")

flow_variable_index_df.reset_index(drop=True, inplace=True)
if len(flow_variable_index_df) != 16_710_231:
    raise ValueError(
        f"Unexpected material-flow variable count: {len(flow_variable_index_df)}"
    )

print(f"Initial material-flow variable count: {len(flow_variable_index_df)}")

In [ ]:
# Transport carbon intensities are expressed in kg CO2-eq per tonne-kilometer.
ROAD_TRANSPORT_CARBON_INTENSITY_KGCO2E_PER_TKM = 0.3333
WATERBORNE_TRANSPORT_CARBON_INTENSITY_KGCO2E_PER_TKM = 0.0374

# Road transport is used except on NDCWV->NDCWP waterborne arcs.
transport_carbon_impact_kgco2e_per_t = np.where(
    flow_variable_index_df["arc_type"] != "NDCWV->NDCWP",
    flow_variable_index_df["distance_km"]
    * ROAD_TRANSPORT_CARBON_INTENSITY_KGCO2E_PER_TKM,
    flow_variable_index_df["distance_km"]
    * WATERBORNE_TRANSPORT_CARBON_INTENSITY_KGCO2E_PER_TKM,
)

In [ ]:
# Assign terminal-material carbon impacts only to arcs entering construction projects.
terminal_material_carbon_impact_kgco2e_per_t_by_code = {
    "CEM": 602.82,
    "NS": 4.76,
    "NCA": 11.95,
    "WAT": 0,
    "RCS": 10.86 + (1.26 - 23.00) / 2.4,
    "RCL": 10.98 + (1.26 - 23.00) / 2.4,
    "RCC": 14.71 - 3.56,
    "RCF": 17.08 - 19.44,
    "RBC": 6.31,
    "RBF": 8.68,
    "RBP": 405.23,
    "WC": 0,
    "WB": 0,
}

terminal_material_carbon_impact_kgco2e_per_t = np.zeros(
    len(flow_variable_index_df)
)
for (
    material_code,
    unit_carbon_impact_kgco2e_per_t,
) in terminal_material_carbon_impact_kgco2e_per_t_by_code.items():
    terminal_material_carbon_impact_kgco2e_per_t += np.where(
        (flow_variable_index_df["destination_node_type"] == "NDPRJ")
        & (flow_variable_index_df["material_code"] == material_code),
        unit_carbon_impact_kgco2e_per_t,
        0,
    )

print(
    "Terminal-material carbon impacts assigned for "
    f"{len(terminal_material_carbon_impact_kgco2e_per_t_by_code)} material codes."
)

In [ ]:
# Component values and conversion factors follow Supplementary Methods 5.
# Both pathway adjustments are expressed in kg CO2-eq per tonne.
concrete_pathway_carbon_adjustment_kgco2e_per_t = np.where(
    flow_variable_index_df["arc_type"] == "NDCBP->NDPRJ",
    (0.82 + 1.68 + 0.61 - 30.18) / 2.4,
    0,
)

block_pathway_carbon_adjustment_kgco2e_per_t = np.where(
    flow_variable_index_df["arc_type"] == "NDBMP->NDPRJ",
    (4.35 + 0.56) - 26.5 / 2.4,
    0,
)

In [ ]:
# Combine transport, terminal-material, and pathway-specific carbon terms.
flow_variable_index_df["unit_carbon_impact_kgco2e_per_t"] = (
    transport_carbon_impact_kgco2e_per_t
    + terminal_material_carbon_impact_kgco2e_per_t
    + concrete_pathway_carbon_adjustment_kgco2e_per_t
    + block_pathway_carbon_adjustment_kgco2e_per_t
)

In [ ]:
def build_node_registry(flow_variable_index_df):
    """Build a sorted node registry from flow-variable endpoints."""
    origin_node_ids = pd.Series(flow_variable_index_df["origin_node_id"].unique())
    destination_node_ids = pd.Series(
        flow_variable_index_df["destination_node_id"].unique()
    )
    endpoint_node_ids = pd.concat([origin_node_ids, destination_node_ids])
    unique_node_ids = endpoint_node_ids.drop_duplicates().reset_index(drop=True)
    node_types = unique_node_ids.map(canonical_node_type)
    node_sequence_numbers = unique_node_ids.str.rsplit("_", n=1).str[1].astype(int)
    node_registry_df = pd.DataFrame(
        {
            "node_id": unique_node_ids,
            "node_type": node_types,
            "node_number": node_sequence_numbers,
        }
    )
    return node_registry_df.sort_values(
        by=["node_type", "node_number"], ascending=True
    ).reset_index(drop=True)


node_registry_df = build_node_registry(flow_variable_index_df)

print("=== Material-flow construction checks ===")
observed_flow_arc_types = flow_variable_index_df["arc_type"].unique()
print(f"Arc types: {sorted(observed_flow_arc_types)}")

grid_to_recycling_arcs_df = flow_variable_index_df.loc[
    flow_variable_index_df["arc_type"] == "NDGRD->NDWRP"
]
print(f"NDGRD->NDWRP arc count: {len(grid_to_recycling_arcs_df)}")
if len(grid_to_recycling_arcs_df) == 0:
    raise ValueError("No NDGRD->NDWRP arcs were generated for WC and WB.")
print(
    "Materials represented on NDGRD->NDWRP arcs: "
    f"{grid_to_recycling_arcs_df['material_code'].unique()}"
)

flow_variable_count_by_material = flow_variable_index_df["material_code"].value_counts()
for material_code in ["WC", "WB", "RCS", "RCL", "RCC", "RCF", "RBC", "RBF", "RBP"]:
    print(
        f"  {material_code}: "
        f"{flow_variable_count_by_material.get(material_code, 0)} variables"
    )

## 3. Indexed model representation

Reusable index maps connect each flow record to its position in the material-flow
variable array. Recycling-scheme throughput uses a separate plant-by-scheme index
table.

In [ ]:
def build_flow_index_maps(flow_variable_index_df):
    """Build reusable row-index maps for constraint assembly."""
    if "row_id" not in flow_variable_index_df.columns:
        flow_variable_index_df["row_id"] = np.arange(
            len(flow_variable_index_df), dtype=np.int64
        )

    flow_indices_by_destination_and_material = {
        key: group["row_id"].to_numpy()
        for key, group in flow_variable_index_df.groupby(
            ["destination_node_id", "material_code"], sort=False
        )
    }
    flow_indices_by_origin_and_material = {
        key: group["row_id"].to_numpy()
        for key, group in flow_variable_index_df.groupby(
            ["origin_node_id", "material_code"], sort=False
        )
    }
    flow_indices_by_origin_node = {
        origin_node_id: group["row_id"].to_numpy()
        for origin_node_id, group in flow_variable_index_df.groupby(
            "origin_node_id", sort=False
        )
    }
    return (
        flow_indices_by_destination_and_material,
        flow_indices_by_origin_and_material,
        flow_indices_by_origin_node,
    )


def add_scheme_throughput_variables(model, recycling_plant_ids, recyclate_yield_matrix):
    """Create non-negative scheme-throughput variables and plant-level maps."""
    recycling_schemes = recyclate_yield_matrix.index.tolist()
    scheme_variable_records = [
        {
            "recycling_plant_id": recycling_plant_id,
            "recycling_scheme": recycling_scheme,
        }
        for recycling_plant_id in recycling_plant_ids
        for recycling_scheme in recycling_schemes
    ]
    scheme_throughput_index_df = pd.DataFrame(scheme_variable_records)
    scheme_throughput_vars = model.addMVar(
        shape=len(scheme_throughput_index_df),
        lb=0.0,
        vtype=GRB.CONTINUOUS,
        name="scheme_throughput",
    )
    model.update()

    recycling_scheme_codes = scheme_throughput_index_df["recycling_scheme"].to_numpy()
    recycling_plant_id_array = scheme_throughput_index_df["recycling_plant_id"].to_numpy()
    concrete_scheme_mask = np.isin(recycling_scheme_codes, ["CRS-S", "CRS-L", "CRS-C"])
    brick_scheme_mask = np.isin(recycling_scheme_codes, ["BRS-C", "BRS-F", "BRS-P"])

    scheme_indices_by_recycling_plant = {}
    concrete_scheme_indices_by_recycling_plant = {}
    brick_scheme_indices_by_recycling_plant = {}
    for recycling_plant_id in recycling_plant_ids:
        recycling_plant_mask = recycling_plant_id_array == recycling_plant_id
        recycling_plant_scheme_indices = np.where(recycling_plant_mask)[0]
        scheme_indices_by_recycling_plant[
            recycling_plant_id
        ] = recycling_plant_scheme_indices
        concrete_scheme_indices_by_recycling_plant[recycling_plant_id] = (
            recycling_plant_scheme_indices[
                np.where(concrete_scheme_mask[recycling_plant_scheme_indices])[0]
            ]
        )
        brick_scheme_indices_by_recycling_plant[recycling_plant_id] = (
            recycling_plant_scheme_indices[
                np.where(brick_scheme_mask[recycling_plant_scheme_indices])[0]
            ]
        )

    return (
        scheme_throughput_vars,
        scheme_throughput_index_df,
        scheme_indices_by_recycling_plant,
        concrete_scheme_indices_by_recycling_plant,
        brick_scheme_indices_by_recycling_plant,
    )


(
    flow_indices_by_destination_and_material,
    flow_indices_by_origin_and_material,
    flow_indices_by_origin_node,
) = build_flow_index_maps(flow_variable_index_df)


In [ ]:
model = gp.Model("ConstrainedMetabolismModel")

log_status("Adding material-flow variables (MVar)")

num_material_flow_vars = len(flow_variable_index_df)
unit_carbon_impact_coefficients = flow_variable_index_df[
    "unit_carbon_impact_kgco2e_per_t"
].to_numpy(dtype=np.double)
material_flow_lower_bounds = np.zeros(num_material_flow_vars, dtype=np.double)
material_flow_upper_bounds = np.full(
    num_material_flow_vars, GRB.INFINITY, dtype=np.double
)

material_flow_vars = model.addMVar(
    shape=num_material_flow_vars,
    obj=unit_carbon_impact_coefficients,
    lb=material_flow_lower_bounds,
    ub=material_flow_upper_bounds,
    vtype=GRB.CONTINUOUS,
    name="material_flow_t_per_year",
)
model.update()
log_status(f"Material-flow variables added ({num_material_flow_vars} variables)")
model.setObjective(model.getObjective(), GRB.MINIMIZE)


def require_node_parameter(node_id, parameter_name):
    """Return one required parameter and fail if its key or value is absent."""
    if node_id not in node_parameter_matrix_df.index:
        raise KeyError(f"Unknown node ID in parameter lookup: {node_id!r}")
    if parameter_name not in node_parameter_matrix_df.columns:
        raise KeyError(f"Unknown parameter name: {parameter_name!r}")
    value_t_per_year = node_parameter_matrix_df.at[node_id, parameter_name]
    if pd.isna(value_t_per_year):
        raise KeyError(f"Missing parameter {parameter_name!r} for node {node_id!r}")
    return float(value_t_per_year)


construction_project_ids = node_registry_df.loc[
    node_registry_df["node_type"] == "NDPRJ", "node_id"
].tolist()
concrete_batching_plant_ids = node_registry_df.loc[
    node_registry_df["node_type"] == "NDCBP", "node_id"
].tolist()
block_manufacturing_plant_ids = node_registry_df.loc[
    node_registry_df["node_type"] == "NDBMP", "node_id"
].tolist()
recycling_plant_ids = node_registry_df.loc[
    node_registry_df["node_type"] == "NDWRP", "node_id"
].tolist()
demolition_waste_grid_cell_ids = node_registry_df.loc[
    node_registry_df["node_type"] == "NDGRD", "node_id"
].tolist()
cement_plant_ids = node_registry_df.loc[
    node_registry_df["node_type"] == "NDCEM", "node_id"
].tolist()
marine_sand_desalination_plant_ids = node_registry_df.loc[
    node_registry_df["node_type"] == "NDSND", "node_id"
].tolist()
quarry_ids = node_registry_df.loc[
    node_registry_df["node_type"] == "NDQRY", "node_id"
].tolist()

## 4. Constraint groups

The implementation uses the C1–C9 group IDs defined in
[`data/codebook.csv`](../data/codebook.csv): project demand and direct substitution
(C1), batching capacity (C2), concrete mix (C3), block capacity (C4), block mix
(C5), recycling conversion and capacity (C6), waste input (C7), virgin-material
supply (C8), and intermediate-node flow conservation (C9).

In [ ]:
def add_c1_project_demand_constraints():
    """Add C1 project-demand equalities and direct-recyclate replacement limits."""
    log_status("Adding C1 project-demand and direct-recyclate constraints")

    project_ids = construction_project_ids
    num_projects = len(project_ids)
    concrete_demand_values = np.array(
        [
            require_node_parameter(project_id, "concrete_demand")
            for project_id in project_ids
        ]
    )
    block_demand_values = np.array(
        [
            require_node_parameter(project_id, "block_demand")
            for project_id in project_ids
        ]
    )
    project_id_to_index = {
        project_id: project_index
        for project_index, project_id in enumerate(project_ids)
    }

    project_id_set = set(project_ids)
    project_inflow_mask = flow_variable_index_df["destination_node_id"].isin(
        project_id_set
    )
    project_inflow_df = flow_variable_index_df.loc[project_inflow_mask].copy()
    project_inflow_indices = np.where(project_inflow_mask)[0]

    from_batching_plant_mask = project_inflow_df["origin_node_type"] == "NDCBP"
    from_batching_plant_df = project_inflow_df[from_batching_plant_mask]
    from_batching_plant_indices = project_inflow_indices[from_batching_plant_mask]

    from_block_plant_mask = project_inflow_df["origin_node_type"] == "NDBMP"
    from_block_plant_df = project_inflow_df[from_block_plant_mask]
    from_block_plant_indices = project_inflow_indices[from_block_plant_mask]

    direct_rcs_inflow_mask = (
        (project_inflow_df["origin_node_type"] == "NDWRP")
        & (project_inflow_df["material_code"] == "RCS")
    )
    direct_rcs_inflow_df = project_inflow_df[direct_rcs_inflow_mask]
    direct_rcs_inflow_indices = project_inflow_indices[direct_rcs_inflow_mask]

    direct_rcl_inflow_mask = (
        (project_inflow_df["origin_node_type"] == "NDWRP")
        & (project_inflow_df["material_code"] == "RCL")
    )
    direct_rcl_inflow_df = project_inflow_df[direct_rcl_inflow_mask]
    direct_rcl_inflow_indices = project_inflow_indices[direct_rcl_inflow_mask]

    concrete_demand_rows = []
    concrete_demand_columns = []
    concrete_demand_coefficients = []

    if len(from_batching_plant_df) > 0:
        batching_plant_destination_indices = from_batching_plant_df[
            "destination_node_id"
        ].map(project_id_to_index).to_numpy()
        concrete_demand_rows.extend(batching_plant_destination_indices)
        concrete_demand_columns.extend(from_batching_plant_indices)
        concrete_demand_coefficients.extend(
            [1.0] * len(from_batching_plant_indices)
        )

    if len(direct_rcs_inflow_df) > 0:
        rcs_project_indices = direct_rcs_inflow_df["destination_node_id"].map(
            project_id_to_index
        ).to_numpy()
        concrete_demand_rows.extend(rcs_project_indices)
        concrete_demand_columns.extend(direct_rcs_inflow_indices)
        concrete_demand_coefficients.extend(
            [1.0] * len(direct_rcs_inflow_indices)
        )

    if len(direct_rcl_inflow_df) > 0:
        rcl_project_indices = direct_rcl_inflow_df["destination_node_id"].map(
            project_id_to_index
        ).to_numpy()
        concrete_demand_rows.extend(rcl_project_indices)
        concrete_demand_columns.extend(direct_rcl_inflow_indices)
        concrete_demand_coefficients.extend(
            [1.0] * len(direct_rcl_inflow_indices)
        )

    block_demand_rows = []
    block_demand_columns = []
    block_demand_coefficients = []

    if len(from_block_plant_df) > 0:
        block_plant_destination_indices = from_block_plant_df[
            "destination_node_id"
        ].map(project_id_to_index).to_numpy()
        block_demand_rows.extend(block_plant_destination_indices + num_projects)
        block_demand_columns.extend(from_block_plant_indices)
        block_demand_coefficients.extend([1.0] * len(from_block_plant_indices))

    rcs_substitution_rows = []
    rcs_substitution_columns = []
    rcs_substitution_coefficients = []

    if len(direct_rcs_inflow_df) > 0:
        rcs_project_indices = direct_rcs_inflow_df["destination_node_id"].map(
            project_id_to_index
        ).to_numpy()
        rcs_substitution_rows.extend(rcs_project_indices + 2 * num_projects)
        rcs_substitution_columns.extend(direct_rcs_inflow_indices)
        rcs_substitution_coefficients.extend(
            [1.0] * len(direct_rcs_inflow_indices)
        )

    shared_rcs_rcl_substitution_rows = []
    shared_rcs_rcl_substitution_columns = []
    shared_rcs_rcl_substitution_coefficients = []

    if len(direct_rcl_inflow_df) > 0:
        rcl_project_indices = direct_rcl_inflow_df["destination_node_id"].map(
            project_id_to_index
        ).to_numpy()
        shared_rcs_rcl_substitution_rows.extend(
            rcl_project_indices + 3 * num_projects
        )
        shared_rcs_rcl_substitution_columns.extend(direct_rcl_inflow_indices)
        shared_rcs_rcl_substitution_coefficients.extend(
            [1.0] * len(direct_rcl_inflow_indices)
        )

    if len(direct_rcs_inflow_df) > 0:
        rcs_project_indices = direct_rcs_inflow_df["destination_node_id"].map(
            project_id_to_index
        ).to_numpy()
        shared_rcs_rcl_substitution_rows.extend(
            rcs_project_indices + 3 * num_projects
        )
        shared_rcs_rcl_substitution_columns.extend(direct_rcs_inflow_indices)
        shared_rcs_rcl_substitution_coefficients.extend(
            [RCL_MAX_REPLACEMENT_RATIO / RCS_MAX_REPLACEMENT_RATIO]
            * len(direct_rcs_inflow_indices)
        )

    c1_constraint_rows = np.concatenate(
        [
            concrete_demand_rows,
            block_demand_rows,
            rcs_substitution_rows,
            shared_rcs_rcl_substitution_rows,
        ]
    )
    c1_constraint_columns = np.concatenate(
        [
            concrete_demand_columns,
            block_demand_columns,
            rcs_substitution_columns,
            shared_rcs_rcl_substitution_columns,
        ]
    )
    c1_constraint_coefficients = np.concatenate(
        [
            concrete_demand_coefficients,
            block_demand_coefficients,
            rcs_substitution_coefficients,
            shared_rcs_rcl_substitution_coefficients,
        ]
    )

    num_constraints = 4 * num_projects
    c1_constraint_matrix = sp.coo_matrix(
        (
            c1_constraint_coefficients,
            (c1_constraint_rows, c1_constraint_columns),
        ),
        shape=(num_constraints, num_material_flow_vars),
    )

    concrete_demand_right_hand_side = concrete_demand_values
    block_demand_right_hand_side = block_demand_values
    rcs_substitution_right_hand_side = (
        RCS_MAX_REPLACEMENT_RATIO * concrete_demand_values
    )
    shared_rcs_rcl_substitution_right_hand_side = (
        RCL_MAX_REPLACEMENT_RATIO * concrete_demand_values
    )
    c1_right_hand_side = np.concatenate(
        [
            concrete_demand_right_hand_side,
            block_demand_right_hand_side,
            rcs_substitution_right_hand_side,
            shared_rcs_rcl_substitution_right_hand_side,
        ]
    )
    c1_senses = np.array(
        ["="] * num_projects
        + ["="] * num_projects
        + ["<="] * num_projects
        + ["<="] * num_projects
    )
    c1_constraint_names = (
        [f"C1_ConcreteDemand_{project_id}" for project_id in project_ids]
        + [f"C1_BlockDemand_{project_id}" for project_id in project_ids]
        + [f"C1_RCS_Substitution_{project_id}" for project_id in project_ids]
        + [f"C1_RCL_Substitution_{project_id}" for project_id in project_ids]
    )

    model.addMConstr(
        c1_constraint_matrix,
        material_flow_vars,
        c1_senses,
        c1_right_hand_side,
        name=c1_constraint_names,
    )
    log_status(
        f"C1 constraints added ({num_constraints} constraints; "
        f"matrix shape {c1_constraint_matrix.shape})"
    )

In [ ]:
def add_c2_batching_capacity_constraints():
    """Add C2 concrete-batching-plant capacity constraints."""
    log_status("Adding C2 concrete-batching-plant capacity constraints")

    batching_plant_ids = concrete_batching_plant_ids
    num_batching_plants = len(batching_plant_ids)
    batching_capacity_values = np.array(
        [
            require_node_parameter(batching_plant_id, "batching_plant_capacity")
            for batching_plant_id in batching_plant_ids
        ]
    )
    batching_plant_id_to_index = {
        batching_plant_id: plant_index
        for plant_index, batching_plant_id in enumerate(batching_plant_ids)
    }

    target_materials = {"CEM", "NS", "NCA", "WAT", "RCC"}
    batching_plant_outflow_mask = (
        flow_variable_index_df["origin_node_id"].isin(batching_plant_ids)
        & flow_variable_index_df["material_code"].isin(target_materials)
    )
    batching_plant_outflow_indices = np.where(batching_plant_outflow_mask)[0]
    batching_plant_outflow_df = flow_variable_index_df.loc[
        batching_plant_outflow_mask
    ].copy()

    c2_constraint_rows = batching_plant_outflow_df["origin_node_id"].map(
        batching_plant_id_to_index
    ).to_numpy()
    c2_constraint_columns = batching_plant_outflow_indices
    c2_constraint_coefficients = np.ones(
        len(batching_plant_outflow_indices), dtype=np.float64
    )
    c2_constraint_matrix = sp.coo_matrix(
        (
            c2_constraint_coefficients,
            (c2_constraint_rows, c2_constraint_columns),
        ),
        shape=(num_batching_plants, num_material_flow_vars),
    )

    c2_right_hand_side = batching_capacity_values
    c2_senses = ["<="] * num_batching_plants
    c2_constraint_names = [
        f"C2_BatchingCapacity_{batching_plant_id}"
        for batching_plant_id in batching_plant_ids
    ]
    model.addMConstr(
        c2_constraint_matrix,
        material_flow_vars,
        c2_senses,
        c2_right_hand_side,
        name=c2_constraint_names,
    )
    log_status(
        f"C2 constraints added ({num_batching_plants} constraints; "
        f"matrix shape {c2_constraint_matrix.shape})"
    )

In [ ]:
def add_c3_concrete_mix_constraints():
    """Add C3 NDCBP-to-NDPRJ mix equations and the RCC replacement limit."""
    log_status("Adding C3 concrete-mix and RCC replacement constraints")

    batching_to_project_mask = (
        flow_variable_index_df["origin_node_id"].isin(concrete_batching_plant_ids)
        & flow_variable_index_df["destination_node_id"].isin(
            construction_project_ids
        )
        & flow_variable_index_df["material_code"].isin(
            ["CEM", "NS", "NCA", "RCC", "WAT"]
        )
    )
    batching_to_project_df = flow_variable_index_df[batching_to_project_mask]

    flow_index_by_plant_project_material = {}
    for flow_variable_index, flow_record in batching_to_project_df.iterrows():
        plant_project_pair = (
            flow_record["origin_node_id"],
            flow_record["destination_node_id"],
        )
        flow_index_by_plant_project_material.setdefault(plant_project_pair, {})[
            flow_record["material_code"]
        ] = flow_variable_index
    batching_project_pairs = list(flow_index_by_plant_project_material)

    c3_constraint_rows = []
    c3_constraint_columns = []
    c3_constraint_coefficients = []
    c3_senses = []
    c3_right_hand_side = []
    c3_constraint_names = []
    constraint_row_index = 0

    for batching_plant_id, project_id in batching_project_pairs:
        flow_index_by_material = flow_index_by_plant_project_material[
            (batching_plant_id, project_id)
        ]
        cement_variable_index = flow_index_by_material.get("CEM")
        natural_sand_variable_index = flow_index_by_material.get("NS")
        natural_coarse_aggregate_variable_index = flow_index_by_material.get("NCA")
        rcc_variable_index = flow_index_by_material.get("RCC")
        water_variable_index = flow_index_by_material.get("WAT")

        if cement_variable_index is not None or natural_sand_variable_index is not None:
            if cement_variable_index is not None:
                c3_constraint_columns.append(cement_variable_index)
                c3_constraint_coefficients.append(-CONCRETE_NATURAL_SAND_MASS_RATIO)
                c3_constraint_rows.append(constraint_row_index)
            if natural_sand_variable_index is not None:
                c3_constraint_columns.append(natural_sand_variable_index)
                c3_constraint_coefficients.append(CONCRETE_CEMENT_MASS_RATIO)
                c3_constraint_rows.append(constraint_row_index)
            c3_senses.append("=")
            c3_right_hand_side.append(0)
            c3_constraint_names.append(
                f"C3_EQ1_{batching_plant_id}_{project_id}"
            )
            constraint_row_index += 1

        if (
            cement_variable_index is not None
            or natural_coarse_aggregate_variable_index is not None
            or rcc_variable_index is not None
        ):
            if cement_variable_index is not None:
                c3_constraint_columns.append(cement_variable_index)
                c3_constraint_coefficients.append(
                    -CONCRETE_COARSE_AGGREGATE_MASS_RATIO
                )
                c3_constraint_rows.append(constraint_row_index)
            if natural_coarse_aggregate_variable_index is not None:
                c3_constraint_columns.append(natural_coarse_aggregate_variable_index)
                c3_constraint_coefficients.append(CONCRETE_CEMENT_MASS_RATIO)
                c3_constraint_rows.append(constraint_row_index)
            if rcc_variable_index is not None:
                c3_constraint_columns.append(rcc_variable_index)
                c3_constraint_coefficients.append(CONCRETE_CEMENT_MASS_RATIO)
                c3_constraint_rows.append(constraint_row_index)
            c3_senses.append("=")
            c3_right_hand_side.append(0)
            c3_constraint_names.append(
                f"C3_EQ2_{batching_plant_id}_{project_id}"
            )
            constraint_row_index += 1

        if cement_variable_index is not None or water_variable_index is not None:
            if cement_variable_index is not None:
                c3_constraint_columns.append(cement_variable_index)
                c3_constraint_coefficients.append(-CONCRETE_WATER_MASS_RATIO)
                c3_constraint_rows.append(constraint_row_index)
            if water_variable_index is not None:
                c3_constraint_columns.append(water_variable_index)
                c3_constraint_coefficients.append(CONCRETE_CEMENT_MASS_RATIO)
                c3_constraint_rows.append(constraint_row_index)
            c3_senses.append("=")
            c3_right_hand_side.append(0)
            c3_constraint_names.append(
                f"C3_EQ3_{batching_plant_id}_{project_id}"
            )
            constraint_row_index += 1

        if rcc_variable_index is not None:
            c3_constraint_columns.append(rcc_variable_index)
            c3_constraint_coefficients.append(
                1 - RCC_MAX_COARSE_AGG_REPLACEMENT_RATIO
            )
            c3_constraint_rows.append(constraint_row_index)
            if natural_coarse_aggregate_variable_index is not None:
                c3_constraint_columns.append(natural_coarse_aggregate_variable_index)
                c3_constraint_coefficients.append(
                    -RCC_MAX_COARSE_AGG_REPLACEMENT_RATIO
                )
                c3_constraint_rows.append(constraint_row_index)
            c3_senses.append("<=")
            c3_right_hand_side.append(0)
            c3_constraint_names.append(
                f"C3_INEQ_{batching_plant_id}_{project_id}"
            )
            constraint_row_index += 1

    if constraint_row_index > 0:
        c3_constraint_matrix = sp.coo_matrix(
            (
                c3_constraint_coefficients,
                (c3_constraint_rows, c3_constraint_columns),
            ),
            shape=(constraint_row_index, num_material_flow_vars),
        )
        model.addMConstr(
            c3_constraint_matrix,
            material_flow_vars,
            c3_senses,
            c3_right_hand_side,
            c3_constraint_names,
        )
        log_status(
            f"C3 constraints added ({constraint_row_index} constraints; "
            f"matrix shape {c3_constraint_matrix.shape})"
        )
    else:
        log_status("No applicable C3 constraints were generated")

In [ ]:
def add_c4_block_plant_capacity_constraints():
    """Add C4 block-manufacturing-plant capacity constraints."""
    log_status("Adding C4 block-manufacturing-plant capacity constraints")

    block_plant_ids = block_manufacturing_plant_ids
    num_block_plants = len(block_plant_ids)
    block_capacity_values = np.array(
        [
            require_node_parameter(block_plant_id, "block_plant_capacity")
            for block_plant_id in block_plant_ids
        ]
    )
    block_plant_id_to_index = {
        block_plant_id: plant_index
        for plant_index, block_plant_id in enumerate(block_plant_ids)
    }

    target_materials = {"CEM", "NS", "NCA", "WAT", "RCF", "RBC", "RBF", "RBP"}
    block_plant_outflow_mask = (
        flow_variable_index_df["origin_node_id"].isin(block_plant_ids)
        & flow_variable_index_df["material_code"].isin(target_materials)
    )
    block_plant_outflow_indices = np.where(block_plant_outflow_mask)[0]
    block_plant_outflow_df = flow_variable_index_df.loc[
        block_plant_outflow_mask
    ].copy()

    c4_constraint_rows = block_plant_outflow_df["origin_node_id"].map(
        block_plant_id_to_index
    ).to_numpy()
    c4_constraint_columns = block_plant_outflow_indices
    c4_constraint_coefficients = np.ones(
        len(block_plant_outflow_indices), dtype=np.float64
    )
    c4_constraint_matrix = sp.coo_matrix(
        (
            c4_constraint_coefficients,
            (c4_constraint_rows, c4_constraint_columns),
        ),
        shape=(num_block_plants, num_material_flow_vars),
    )

    c4_right_hand_side = block_capacity_values
    c4_senses = ["<="] * num_block_plants
    c4_constraint_names = [
        f"C4_BlockCapacity_{block_plant_id}" for block_plant_id in block_plant_ids
    ]
    model.addMConstr(
        c4_constraint_matrix,
        material_flow_vars,
        c4_senses,
        c4_right_hand_side,
        name=c4_constraint_names,
    )
    log_status(
        f"C4 constraints added ({num_block_plants} constraints; "
        f"matrix shape {c4_constraint_matrix.shape})"
    )

In [ ]:
def add_c5_block_mix_constraints():
    """Add C5 NDBMP-to-NDPRJ mix equations and recyclate replacement limits."""
    log_status("Adding C5 block-mix and recyclate replacement constraints")

    block_plant_ids = block_manufacturing_plant_ids
    project_ids = construction_project_ids
    target_materials = {"CEM", "NS", "NCA", "WAT", "RCF", "RBC", "RBF", "RBP"}
    block_to_project_mask = (
        flow_variable_index_df["origin_node_id"].isin(set(block_plant_ids))
        & flow_variable_index_df["destination_node_id"].isin(set(project_ids))
        & flow_variable_index_df["material_code"].isin(target_materials)
    )
    block_to_project_indices = np.where(block_to_project_mask)[0]
    block_to_project_df = flow_variable_index_df.loc[block_to_project_mask].copy()

    if len(block_to_project_df) == 0:
        log_status("No applicable C5 constraints were generated")
        return

    block_to_project_pairs_df = block_to_project_df[
        ["origin_node_id", "destination_node_id"]
    ].drop_duplicates()
    plant_project_pairs = [
        (row["origin_node_id"], row["destination_node_id"])
        for _, row in block_to_project_pairs_df.iterrows()
    ]
    num_plant_project_pairs = len(plant_project_pairs)
    plant_project_pair_to_index = {
        pair: pair_index for pair_index, pair in enumerate(plant_project_pairs)
    }

    block_to_project_df["plant_project_pair"] = list(
        zip(
            block_to_project_df["origin_node_id"],
            block_to_project_df["destination_node_id"],
        )
    )
    block_to_project_df["plant_project_pair_index"] = block_to_project_df[
        "plant_project_pair"
    ].map(plant_project_pair_to_index)

    flow_indices_by_material = {}
    for material_code in target_materials:
        material_mask = (
            block_to_project_df["material_code"] == material_code
        ).to_numpy()
        if material_mask.any():
            flow_indices_by_material[material_code] = {
                "flow_variable_indices": block_to_project_indices[material_mask],
                "plant_project_pair_indices": block_to_project_df.loc[
                    material_mask, "plant_project_pair_index"
                ].to_numpy(),
            }
        else:
            flow_indices_by_material[material_code] = {
                "flow_variable_indices": np.array([]),
                "plant_project_pair_indices": np.array([]),
            }

    num_constraints_per_pair = 6
    total_constraints = num_plant_project_pairs * num_constraints_per_pair
    c5_constraint_rows = []
    c5_constraint_columns = []
    c5_constraint_coefficients = []

    cement_and_brick_powder_mass_ratio = float(
        BLOCK_CEMENT_AND_BRICK_POWDER_MASS_RATIO
    )
    fine_aggregate_mass_ratio = float(BLOCK_FINE_AGGREGATE_MASS_RATIO)
    coarse_aggregate_mass_ratio = float(BLOCK_COARSE_AGGREGATE_MASS_RATIO)
    water_mass_ratio = float(BLOCK_WATER_MASS_RATIO)

    cement_flow_data = flow_indices_by_material["CEM"]
    natural_sand_flow_data = flow_indices_by_material["NS"]
    natural_coarse_aggregate_flow_data = flow_indices_by_material["NCA"]
    water_flow_data = flow_indices_by_material["WAT"]
    rcf_flow_data = flow_indices_by_material["RCF"]
    rbc_flow_data = flow_indices_by_material["RBC"]
    rbf_flow_data = flow_indices_by_material["RBF"]
    rbp_flow_data = flow_indices_by_material["RBP"]

    constraint_offset = 0
    for material_flow_data in [cement_flow_data, rbp_flow_data]:
        for flow_variable_index, plant_project_pair_index in zip(
            material_flow_data["flow_variable_indices"],
            material_flow_data["plant_project_pair_indices"],
        ):
            c5_constraint_rows.append(
                plant_project_pair_index
                + constraint_offset * num_plant_project_pairs
            )
            c5_constraint_columns.append(flow_variable_index)
            c5_constraint_coefficients.append(fine_aggregate_mass_ratio)

    for material_flow_data in [
        natural_sand_flow_data,
        rcf_flow_data,
        rbf_flow_data,
    ]:
        for flow_variable_index, plant_project_pair_index in zip(
            material_flow_data["flow_variable_indices"],
            material_flow_data["plant_project_pair_indices"],
        ):
            c5_constraint_rows.append(
                plant_project_pair_index
                + constraint_offset * num_plant_project_pairs
            )
            c5_constraint_columns.append(flow_variable_index)
            c5_constraint_coefficients.append(
                -cement_and_brick_powder_mass_ratio
            )

    constraint_offset = 1
    for material_flow_data in [cement_flow_data, rbp_flow_data]:
        for flow_variable_index, plant_project_pair_index in zip(
            material_flow_data["flow_variable_indices"],
            material_flow_data["plant_project_pair_indices"],
        ):
            c5_constraint_rows.append(
                plant_project_pair_index
                + constraint_offset * num_plant_project_pairs
            )
            c5_constraint_columns.append(flow_variable_index)
            c5_constraint_coefficients.append(coarse_aggregate_mass_ratio)

    for material_flow_data in [natural_coarse_aggregate_flow_data, rbc_flow_data]:
        for flow_variable_index, plant_project_pair_index in zip(
            material_flow_data["flow_variable_indices"],
            material_flow_data["plant_project_pair_indices"],
        ):
            c5_constraint_rows.append(
                plant_project_pair_index
                + constraint_offset * num_plant_project_pairs
            )
            c5_constraint_columns.append(flow_variable_index)
            c5_constraint_coefficients.append(
                -cement_and_brick_powder_mass_ratio
            )

    constraint_offset = 2
    for material_flow_data in [cement_flow_data, rbp_flow_data]:
        for flow_variable_index, plant_project_pair_index in zip(
            material_flow_data["flow_variable_indices"],
            material_flow_data["plant_project_pair_indices"],
        ):
            c5_constraint_rows.append(
                plant_project_pair_index
                + constraint_offset * num_plant_project_pairs
            )
            c5_constraint_columns.append(flow_variable_index)
            c5_constraint_coefficients.append(water_mass_ratio)

    for flow_variable_index, plant_project_pair_index in zip(
        water_flow_data["flow_variable_indices"],
        water_flow_data["plant_project_pair_indices"],
    ):
        c5_constraint_rows.append(
            plant_project_pair_index + constraint_offset * num_plant_project_pairs
        )
        c5_constraint_columns.append(flow_variable_index)
        c5_constraint_coefficients.append(-cement_and_brick_powder_mass_ratio)

    constraint_offset = 3
    for flow_variable_index, plant_project_pair_index in zip(
        natural_sand_flow_data["flow_variable_indices"],
        natural_sand_flow_data["plant_project_pair_indices"],
    ):
        c5_constraint_rows.append(
            plant_project_pair_index + constraint_offset * num_plant_project_pairs
        )
        c5_constraint_columns.append(flow_variable_index)
        c5_constraint_coefficients.append(
            -RCF_RBF_MAX_FINE_AGG_REPLACEMENT_RATIO
        )

    for material_flow_data in [rcf_flow_data, rbf_flow_data]:
        for flow_variable_index, plant_project_pair_index in zip(
            material_flow_data["flow_variable_indices"],
            material_flow_data["plant_project_pair_indices"],
        ):
            c5_constraint_rows.append(
                plant_project_pair_index
                + constraint_offset * num_plant_project_pairs
            )
            c5_constraint_columns.append(flow_variable_index)
            c5_constraint_coefficients.append(
                1 - RCF_RBF_MAX_FINE_AGG_REPLACEMENT_RATIO
            )

    constraint_offset = 4
    for flow_variable_index, plant_project_pair_index in zip(
        natural_coarse_aggregate_flow_data["flow_variable_indices"],
        natural_coarse_aggregate_flow_data["plant_project_pair_indices"],
    ):
        c5_constraint_rows.append(
            plant_project_pair_index + constraint_offset * num_plant_project_pairs
        )
        c5_constraint_columns.append(flow_variable_index)
        c5_constraint_coefficients.append(
            -RBC_MAX_COARSE_AGG_REPLACEMENT_RATIO
        )

    for flow_variable_index, plant_project_pair_index in zip(
        rbc_flow_data["flow_variable_indices"],
        rbc_flow_data["plant_project_pair_indices"],
    ):
        c5_constraint_rows.append(
            plant_project_pair_index + constraint_offset * num_plant_project_pairs
        )
        c5_constraint_columns.append(flow_variable_index)
        c5_constraint_coefficients.append(1 - RBC_MAX_COARSE_AGG_REPLACEMENT_RATIO)

    constraint_offset = 5
    for flow_variable_index, plant_project_pair_index in zip(
        rbp_flow_data["flow_variable_indices"],
        rbp_flow_data["plant_project_pair_indices"],
    ):
        c5_constraint_rows.append(
            plant_project_pair_index + constraint_offset * num_plant_project_pairs
        )
        c5_constraint_columns.append(flow_variable_index)
        c5_constraint_coefficients.append(1 - RBP_MAX_CEMENT_REPLACEMENT_RATIO)

    for flow_variable_index, plant_project_pair_index in zip(
        cement_flow_data["flow_variable_indices"],
        cement_flow_data["plant_project_pair_indices"],
    ):
        c5_constraint_rows.append(
            plant_project_pair_index + constraint_offset * num_plant_project_pairs
        )
        c5_constraint_columns.append(flow_variable_index)
        c5_constraint_coefficients.append(-RBP_MAX_CEMENT_REPLACEMENT_RATIO)

    if len(c5_constraint_coefficients) == 0:
        log_status("No applicable C5 constraint data were generated")
        return

    c5_constraint_rows = np.asarray(c5_constraint_rows)
    c5_constraint_columns = np.asarray(c5_constraint_columns)
    c5_constraint_coefficients = np.asarray(c5_constraint_coefficients)
    maximum_constraint_row = c5_constraint_rows.max()
    if maximum_constraint_row >= total_constraints:
        total_constraints = maximum_constraint_row + 1

    c5_constraint_matrix = sp.coo_matrix(
        (
            c5_constraint_coefficients,
            (c5_constraint_rows, c5_constraint_columns),
        ),
        shape=(total_constraints, num_material_flow_vars),
    )
    c5_right_hand_side = np.zeros(c5_constraint_matrix.shape[0])
    c5_senses = np.array(
        ["="] * (3 * num_plant_project_pairs)
        + ["<="] * (3 * num_plant_project_pairs)
    )
    if c5_senses.shape[0] < c5_right_hand_side.shape[0]:
        c5_senses = np.concatenate(
            [
                c5_senses,
                ["="] * (c5_right_hand_side.shape[0] - c5_senses.shape[0]),
            ]
        )

    c5_constraint_names = (
        [
            f"C5_FineAggregateMix_{block_plant_id}_{project_id}"
            for block_plant_id, project_id in plant_project_pairs
        ]
        + [
            f"C5_CoarseAggregateMix_{block_plant_id}_{project_id}"
            for block_plant_id, project_id in plant_project_pairs
        ]
        + [
            f"C5_WaterMix_{block_plant_id}_{project_id}"
            for block_plant_id, project_id in plant_project_pairs
        ]
        + [
            f"C5_RCF_RBF_Substitution_{block_plant_id}_{project_id}"
            for block_plant_id, project_id in plant_project_pairs
        ]
        + [
            f"C5_RBC_Substitution_{block_plant_id}_{project_id}"
            for block_plant_id, project_id in plant_project_pairs
        ]
        + [
            f"C5_RBP_Substitution_{block_plant_id}_{project_id}"
            for block_plant_id, project_id in plant_project_pairs
        ]
    )
    if len(c5_constraint_names) != c5_constraint_matrix.shape[0]:
        raise RuntimeError(
            "C5 constraint-name count does not match the matrix row count."
        )

    model.addMConstr(
        c5_constraint_matrix,
        material_flow_vars,
        c5_senses,
        c5_right_hand_side,
        name=c5_constraint_names,
    )
    log_status(
        f"C5 constraints added ({c5_constraint_matrix.shape[0]} constraints; "
        f"{num_plant_project_pairs} plant-project pairs; "
        f"matrix shape {c5_constraint_matrix.shape})"
    )

In [ ]:
def add_c6_recycling_scheme_constraints():
    """Link feedstocks, scheme throughput, fixed yields, outputs, and capacity."""
    log_status("Adding C6 recycling-scheme constraints")

    concrete_recyclate_yields = {
        "RCS": [0.50, 0.00, 0.00],  # CRS-S, CRS-L, CRS-C
        "RCL": [0.31, 0.62, 0.00],
        "RCC": [0.13, 0.25, 0.70],
        "RCF": [0.06, 0.13, 0.30],
    }
    concrete_recyclate_yield_matrix = pd.DataFrame(
        concrete_recyclate_yields,
        index=["CRS-S", "CRS-L", "CRS-C"],
    )

    brick_recyclate_yields = {
        "RBC": [0.7, 0, 0],  # BRS-C, BRS-F, BRS-P
        "RBF": [0.3, 1, 0],
        "RBP": [0.0, 0, 1],
    }
    brick_recyclate_yield_matrix = pd.DataFrame(
        brick_recyclate_yields,
        index=["BRS-C", "BRS-F", "BRS-P"],
    )
    recyclate_yield_matrix = pd.concat(
        [concrete_recyclate_yield_matrix, brick_recyclate_yield_matrix], axis=0
    ).fillna(0.0)

    global scheme_throughput_vars, scheme_throughput_index_df
    (
        scheme_throughput_vars,
        scheme_throughput_index_df,
        scheme_indices_by_recycling_plant,
        concrete_scheme_indices_by_recycling_plant,
        brick_scheme_indices_by_recycling_plant,
    ) = add_scheme_throughput_variables(
        model, recycling_plant_ids, recyclate_yield_matrix
    )

    def sum_selected_variables(variable_array, indices):
        """Return the sum of selected Gurobi variables, or zero for no selection."""
        if len(indices) == 0:
            return gp.LinExpr(0.0)
        return variable_array[indices].sum()

    def weighted_sum_selected_variables(coefficients, variable_array, indices):
        """Return a weighted sum of selected Gurobi variables."""
        if len(indices) == 0:
            return gp.LinExpr(0.0)
        coefficients = np.asarray(coefficients, dtype=float)
        if coefficients.size == 0:
            return gp.LinExpr(0.0)
        return coefficients @ variable_array[indices]

    model.addConstrs(
        (
            sum_selected_variables(
                material_flow_vars,
                flow_indices_by_destination_and_material.get(
                    (recycling_plant_id, "WC"), []
                ),
            )
            - sum_selected_variables(
                scheme_throughput_vars,
                concrete_scheme_indices_by_recycling_plant[recycling_plant_id],
            )
            == 0
            for recycling_plant_id in recycling_plant_ids
        ),
        name="C6_WC_Input",
    )
    model.addConstrs(
        (
            sum_selected_variables(
                material_flow_vars,
                flow_indices_by_destination_and_material.get(
                    (recycling_plant_id, "WB"), []
                ),
            )
            - sum_selected_variables(
                scheme_throughput_vars,
                brick_scheme_indices_by_recycling_plant[recycling_plant_id],
            )
            == 0
            for recycling_plant_id in recycling_plant_ids
        ),
        name="C6_WB_Input",
    )

    output_materials = ["RCS", "RCL", "RCC", "RCF", "RBC", "RBF", "RBP"]
    model.addConstrs(
        (
            sum_selected_variables(
                material_flow_vars,
                flow_indices_by_origin_and_material.get(
                    (recycling_plant_id, material_code), []
                ),
            )
            - weighted_sum_selected_variables(
                recyclate_yield_matrix.loc[
                    scheme_throughput_index_df.iloc[
                        scheme_indices_by_recycling_plant[recycling_plant_id]
                    ]["recycling_scheme"].to_numpy(),
                    material_code,
                ].to_numpy(dtype=float),
                scheme_throughput_vars,
                scheme_indices_by_recycling_plant[recycling_plant_id],
            )
            == 0
            for recycling_plant_id in recycling_plant_ids
            for material_code in output_materials
        ),
        name="C6_Output",
    )

    model.addConstrs(
        (
            sum_selected_variables(
                material_flow_vars,
                flow_indices_by_origin_node.get(recycling_plant_id, []),
            )
            <= float(
                require_node_parameter(
                    recycling_plant_id, "recycling_plant_capacity"
                )
            )
            for recycling_plant_id in recycling_plant_ids
        ),
        name="C6_Capacity",
    )
    log_status("C6 recycling-scheme constraints added")

In [ ]:
def add_c7_waste_feedstock_input_constraints():
    """Set WC and WB outflows equal to the registered scenario inputs."""
    log_status("Adding C7 waste-feedstock input constraints")

    grid_cell_ids = demolition_waste_grid_cell_ids
    num_grid_cells = len(grid_cell_ids)
    waste_concrete_input_values = np.array(
        [
            float(waste_concrete_input_by_grid_cell.at[grid_cell_id])
            for grid_cell_id in grid_cell_ids
        ]
    )
    waste_brick_input_values = np.array(
        [
            float(waste_brick_input_by_grid_cell.at[grid_cell_id])
            for grid_cell_id in grid_cell_ids
        ]
    )
    grid_cell_id_to_index = {
        grid_cell_id: grid_cell_index
        for grid_cell_index, grid_cell_id in enumerate(grid_cell_ids)
    }

    waste_concrete_outflow_mask = (
        flow_variable_index_df["origin_node_id"].isin(grid_cell_ids)
        & (flow_variable_index_df["material_code"] == "WC")
    )
    waste_concrete_flow_indices = np.where(waste_concrete_outflow_mask)[0]
    waste_concrete_flow_df = flow_variable_index_df[
        waste_concrete_outflow_mask
    ].copy()

    waste_brick_outflow_mask = (
        flow_variable_index_df["origin_node_id"].isin(grid_cell_ids)
        & (flow_variable_index_df["material_code"] == "WB")
    )
    waste_brick_flow_indices = np.where(waste_brick_outflow_mask)[0]
    waste_brick_flow_df = flow_variable_index_df[waste_brick_outflow_mask].copy()

    if len(waste_concrete_flow_indices) > 0:
        waste_concrete_constraint_rows = waste_concrete_flow_df[
            "origin_node_id"
        ].map(grid_cell_id_to_index).to_numpy()
        waste_concrete_constraint_columns = waste_concrete_flow_indices
        waste_concrete_constraint_coefficients = np.ones(
            len(waste_concrete_flow_indices), dtype=np.float64
        )
        waste_concrete_constraint_matrix = sp.coo_matrix(
            (
                waste_concrete_constraint_coefficients,
                (
                    waste_concrete_constraint_rows,
                    waste_concrete_constraint_columns,
                ),
            ),
            shape=(num_grid_cells, num_material_flow_vars),
        )
        model.addMConstr(
            waste_concrete_constraint_matrix,
            material_flow_vars,
            "=",
            waste_concrete_input_values,
            name=[
                f"C7_WasteConcreteInput_{grid_cell_id}"
                for grid_cell_id in grid_cell_ids
            ],
        )

    if len(waste_brick_flow_indices) > 0:
        waste_brick_constraint_rows = waste_brick_flow_df["origin_node_id"].map(
            grid_cell_id_to_index
        ).to_numpy()
        waste_brick_constraint_columns = waste_brick_flow_indices
        waste_brick_constraint_coefficients = np.ones(
            len(waste_brick_flow_indices), dtype=np.float64
        )
        waste_brick_constraint_matrix = sp.coo_matrix(
            (
                waste_brick_constraint_coefficients,
                (waste_brick_constraint_rows, waste_brick_constraint_columns),
            ),
            shape=(num_grid_cells, num_material_flow_vars),
        )
        model.addMConstr(
            waste_brick_constraint_matrix,
            material_flow_vars,
            "=",
            waste_brick_input_values,
            name=[
                f"C7_WasteBrickInput_{grid_cell_id}"
                for grid_cell_id in grid_cell_ids
            ],
        )

    log_status(
        f"C7 constraints added for {num_grid_cells} demolition-waste grid cells"
    )

In [ ]:
def add_c8_virgin_supply_constraints():
    """Add C8 capacity constraints for virgin-material suppliers."""
    log_status("Adding C8 virgin-material supply-capacity constraints")

    cement_supplier_ids = cement_plant_ids
    natural_sand_supplier_ids = marine_sand_desalination_plant_ids
    coarse_aggregate_supplier_ids = quarry_ids
    virgin_supplier_ids = (
        cement_supplier_ids
        + natural_sand_supplier_ids
        + coarse_aggregate_supplier_ids
    )
    num_virgin_suppliers = len(virgin_supplier_ids)
    virgin_supplier_id_to_index = {
        virgin_supplier_id: supplier_index
        for supplier_index, virgin_supplier_id in enumerate(virgin_supplier_ids)
    }

    virgin_supplier_outflow_mask = flow_variable_index_df["origin_node_id"].isin(
        set(virgin_supplier_ids)
    )
    virgin_supplier_outflow_indices = np.where(virgin_supplier_outflow_mask)[0]
    virgin_supplier_outflow_df = flow_variable_index_df.loc[
        virgin_supplier_outflow_mask
    ].copy()

    cement_mask = (
        (virgin_supplier_outflow_df["material_code"] == "CEM")
        & virgin_supplier_outflow_df["origin_node_id"].isin(cement_supplier_ids)
    )
    cement_variable_indices = virgin_supplier_outflow_indices[
        np.where(cement_mask)[0]
    ]
    cement_constraint_rows = virgin_supplier_outflow_df.loc[
        cement_mask, "origin_node_id"
    ].map(virgin_supplier_id_to_index).to_numpy()
    cement_constraint_columns = cement_variable_indices
    cement_constraint_coefficients = np.ones(
        len(cement_variable_indices), dtype=np.float64
    )

    natural_sand_mask = (
        (virgin_supplier_outflow_df["material_code"] == "NS")
        & virgin_supplier_outflow_df["origin_node_id"].isin(
            natural_sand_supplier_ids
        )
    )
    natural_sand_variable_indices = virgin_supplier_outflow_indices[
        np.where(natural_sand_mask)[0]
    ]
    natural_sand_constraint_rows = virgin_supplier_outflow_df.loc[
        natural_sand_mask, "origin_node_id"
    ].map(virgin_supplier_id_to_index).to_numpy()
    natural_sand_constraint_columns = natural_sand_variable_indices
    natural_sand_constraint_coefficients = np.ones(
        len(natural_sand_variable_indices), dtype=np.float64
    )

    natural_coarse_aggregate_mask = (
        (virgin_supplier_outflow_df["material_code"] == "NCA")
        & virgin_supplier_outflow_df["origin_node_id"].isin(
            coarse_aggregate_supplier_ids
        )
    )
    natural_coarse_aggregate_variable_indices = virgin_supplier_outflow_indices[
        np.where(natural_coarse_aggregate_mask)[0]
    ]
    natural_coarse_aggregate_constraint_rows = virgin_supplier_outflow_df.loc[
        natural_coarse_aggregate_mask, "origin_node_id"
    ].map(virgin_supplier_id_to_index).to_numpy()
    natural_coarse_aggregate_constraint_columns = (
        natural_coarse_aggregate_variable_indices
    )
    natural_coarse_aggregate_constraint_coefficients = np.ones(
        len(natural_coarse_aggregate_variable_indices), dtype=np.float64
    )

    c8_constraint_rows = np.concatenate(
        [
            cement_constraint_rows,
            natural_sand_constraint_rows,
            natural_coarse_aggregate_constraint_rows,
        ]
    )
    c8_constraint_columns = np.concatenate(
        [
            cement_constraint_columns,
            natural_sand_constraint_columns,
            natural_coarse_aggregate_constraint_columns,
        ]
    )
    c8_constraint_coefficients = np.concatenate(
        [
            cement_constraint_coefficients,
            natural_sand_constraint_coefficients,
            natural_coarse_aggregate_constraint_coefficients,
        ]
    )
    c8_constraint_matrix = sp.coo_matrix(
        (
            c8_constraint_coefficients,
            (c8_constraint_rows, c8_constraint_columns),
        ),
        shape=(num_virgin_suppliers, num_material_flow_vars),
    )

    c8_right_hand_side = np.zeros(num_virgin_suppliers, dtype=np.float64)
    for virgin_supplier_id in cement_supplier_ids:
        supplier_index = virgin_supplier_id_to_index[virgin_supplier_id]
        c8_right_hand_side[supplier_index] = require_node_parameter(
            virgin_supplier_id, "cement_supply_capacity"
        )
    for virgin_supplier_id in natural_sand_supplier_ids:
        supplier_index = virgin_supplier_id_to_index[virgin_supplier_id]
        c8_right_hand_side[supplier_index] = require_node_parameter(
            virgin_supplier_id, "marine_sand_supply_capacity"
        )
    for virgin_supplier_id in coarse_aggregate_supplier_ids:
        supplier_index = virgin_supplier_id_to_index[virgin_supplier_id]
        c8_right_hand_side[supplier_index] = require_node_parameter(
            virgin_supplier_id, "natural_coarse_aggregate_supply_capacity"
        )

    c8_senses = np.array(["<="] * num_virgin_suppliers)
    c8_constraint_names = []
    for virgin_supplier_id in virgin_supplier_ids:
        if virgin_supplier_id in cement_supplier_ids:
            c8_constraint_names.append(f"C8_CementSupply_{virgin_supplier_id}")
        elif virgin_supplier_id in natural_sand_supplier_ids:
            c8_constraint_names.append(
                f"C8_NaturalSandSupply_{virgin_supplier_id}"
            )
        else:
            c8_constraint_names.append(
                f"C8_CoarseAggregateSupply_{virgin_supplier_id}"
            )

    model.addMConstr(
        c8_constraint_matrix,
        material_flow_vars,
        c8_senses,
        c8_right_hand_side,
        name=c8_constraint_names,
    )
    log_status(
        f"C8 constraints added ({num_virgin_suppliers} constraints; "
        f"matrix shape {c8_constraint_matrix.shape})"
    )

In [ ]:
def add_c9_flow_balance_constraints():
    """Add C9 material-flow conservation at intermediate nodes."""
    log_status("Adding C9 intermediate-node material-flow constraints")

    intermediate_node_types = ["NDCWV", "NDCWP", "NDCBP", "NDBMP"]
    intermediate_node_ids = []
    node_type_by_id = {}
    for node_type in intermediate_node_types:
        node_ids_of_type = node_registry_df.loc[
            node_registry_df["node_type"] == node_type, "node_id"
        ].tolist()
        for node_id in node_ids_of_type:
            intermediate_node_ids.append(node_id)
            node_type_by_id[node_id] = node_type

    material_codes = flow_variable_index_df["material_code"].unique().tolist()
    valid_node_material_pairs = []
    for node_id in intermediate_node_ids:
        node_type = node_type_by_id[node_id]
        for material_code in material_codes:
            if material_code == "WAT" and node_type in {"NDCBP", "NDBMP"}:
                continue
            valid_node_material_pairs.append((node_id, material_code))

    num_constraints = len(valid_node_material_pairs)
    constraint_row_by_node_material = {
        pair: constraint_row
        for constraint_row, pair in enumerate(valid_node_material_pairs)
    }

    intermediate_node_id_set = set(intermediate_node_ids)
    inflow_mask = flow_variable_index_df["destination_node_id"].isin(
        intermediate_node_id_set
    )
    outflow_mask = flow_variable_index_df["origin_node_id"].isin(
        intermediate_node_id_set
    )
    inflow_indices = np.where(inflow_mask)[0]
    outflow_indices = np.where(outflow_mask)[0]

    c9_constraint_rows = []
    c9_constraint_columns = []
    c9_constraint_coefficients = []

    inflow_node_ids = flow_variable_index_df.loc[
        inflow_mask, "destination_node_id"
    ]
    inflow_material_codes = flow_variable_index_df.loc[inflow_mask, "material_code"]
    inflow_constraint_keys = list(zip(inflow_node_ids, inflow_material_codes))
    valid_inflow_mask = np.array(
        [key in constraint_row_by_node_material for key in inflow_constraint_keys]
    )
    valid_inflow_indices = np.where(valid_inflow_mask)[0]
    if len(valid_inflow_indices) > 0:
        inflow_constraint_rows = [
            constraint_row_by_node_material[inflow_constraint_keys[flow_index]]
            for flow_index in valid_inflow_indices
        ]
        c9_constraint_rows.extend(inflow_constraint_rows)
        c9_constraint_columns.extend(inflow_indices[valid_inflow_indices])
        c9_constraint_coefficients.extend([1.0] * len(valid_inflow_indices))

    outflow_node_ids = flow_variable_index_df.loc[outflow_mask, "origin_node_id"]
    outflow_material_codes = flow_variable_index_df.loc[
        outflow_mask, "material_code"
    ]
    outflow_constraint_keys = list(zip(outflow_node_ids, outflow_material_codes))
    valid_outflow_mask = np.array(
        [key in constraint_row_by_node_material for key in outflow_constraint_keys]
    )
    valid_outflow_indices = np.where(valid_outflow_mask)[0]
    if len(valid_outflow_indices) > 0:
        outflow_constraint_rows = [
            constraint_row_by_node_material[outflow_constraint_keys[flow_index]]
            for flow_index in valid_outflow_indices
        ]
        c9_constraint_rows.extend(outflow_constraint_rows)
        c9_constraint_columns.extend(outflow_indices[valid_outflow_indices])
        c9_constraint_coefficients.extend([-1.0] * len(valid_outflow_indices))

    c9_constraint_matrix = sp.coo_matrix(
        (
            c9_constraint_coefficients,
            (c9_constraint_rows, c9_constraint_columns),
        ),
        shape=(num_constraints, num_material_flow_vars),
    )
    c9_right_hand_side = np.zeros(num_constraints)
    c9_senses = ["="] * num_constraints
    c9_constraint_names = [
        f"C9_FlowConservation_{material_code}_{node_id}"
        for node_id, material_code in valid_node_material_pairs
    ]
    model.addMConstr(
        c9_constraint_matrix,
        material_flow_vars,
        c9_senses,
        c9_right_hand_side,
        name=c9_constraint_names,
    )
    log_status(
        f"C9 constraints added ({num_constraints} constraints; "
        f"matrix shape {c9_constraint_matrix.shape})"
    )

## 5. Add constraint groups

The mapping below controls the C1–C9 groups. The default configuration enables
every group, and the groups are always added in numerical order.

In [ ]:
def add_selected_constraint_groups(constraint_group_switches=None):
    """Validate and add enabled C1–C9 constraint groups in numerical order."""
    constraint_group_order = tuple(f"C{group_number}" for group_number in range(1, 10))
    if constraint_group_switches is None:
        constraint_group_switches = {
            group_id: True for group_id in constraint_group_order
        }
    if not isinstance(constraint_group_switches, dict):
        raise TypeError("constraint_group_switches must be a dictionary or None.")

    observed_group_ids = set(constraint_group_switches)
    expected_group_ids = set(constraint_group_order)
    if observed_group_ids != expected_group_ids:
        missing_group_ids = sorted(expected_group_ids - observed_group_ids)
        unexpected_group_ids = sorted(observed_group_ids - expected_group_ids)
        raise ValueError(
            "Constraint-group switches must contain exactly C1-C9; "
            f"missing={missing_group_ids}, unexpected={unexpected_group_ids}."
        )
    if any(
        type(constraint_group_switches[group_id]) not in (bool, int)
        or constraint_group_switches[group_id] not in (0, 1)
        for group_id in constraint_group_order
    ):
        raise TypeError(
            "Each constraint-group switch must be True, False, 1, or 0."
        )

    constraint_group_functions = {
        "C1": add_c1_project_demand_constraints,
        "C2": add_c2_batching_capacity_constraints,
        "C3": add_c3_concrete_mix_constraints,
        "C4": add_c4_block_plant_capacity_constraints,
        "C5": add_c5_block_mix_constraints,
        "C6": add_c6_recycling_scheme_constraints,
        "C7": add_c7_waste_feedstock_input_constraints,
        "C8": add_c8_virgin_supply_constraints,
        "C9": add_c9_flow_balance_constraints,
    }
    for group_id in constraint_group_order:
        if constraint_group_switches[group_id]:
            log_status(f"Adding constraint group {group_id}")
            constraint_group_functions[group_id]()
        else:
            log_status(f"Constraint group {group_id} skipped")

In [ ]:
ENABLED_CONSTRAINT_GROUPS = {
    "C1": True,
    "C2": True,
    "C3": True,
    "C4": True,
    "C5": True,
    "C6": True,
    "C7": True,
    "C8": True,
    "C9": True,
}
add_selected_constraint_groups(ENABLED_CONSTRAINT_GROUPS)

## 6. Solve the constrained model

The full model requires a valid Gurobi license and substantial memory. Run this
section after the input-validation cells complete successfully. The configurable
time limit is defined near the start of the notebook.

In [ ]:
model.setParam("TimeLimit", TIME_LIMIT_SECONDS)
model.update()

print(f"Variables: {model.NumVars}")
print(f"Constraints: {model.NumConstrs}")
model.optimize()

if model.Status == GRB.OPTIMAL:
    print(
        "\nThe model run completed successfully. "
        f"Total modeled carbon impact: {model.ObjVal:.2f} kg CO2-eq yr^-1"
    )
elif model.Status == GRB.INFEASIBLE:
    print("\nThe model is infeasible. Computing an IIS to identify conflicts...")
    model.computeIIS()
    model.write(INFEASIBILITY_REPORT_PATH)
    print(f"IIS written to {INFEASIBILITY_REPORT_PATH}.")
else:
    print(f"\nModel run stopped with status {model.Status}")

## 7. Diagnostics and exported results

After a successful run, this section attaches solution values, reports material-flow
and carbon-impact checks, and exports the positive-flow and recycling-scheme
throughput tables documented in
[`data/DATA_SCHEMA.md`](../data/DATA_SCHEMA.md). Only flows and scheme throughputs
strictly greater than `REPORTING_THRESHOLD_T_PER_YEAR` are included in the exported
tables.

In [ ]:
def report_solution_diagnostics():
    """Report flow, throughput, demand, and carbon-impact diagnostics."""

    def utilization_text(outflow_t_per_year, input_t_per_year):
        """Format utilization without dividing by a zero network input."""
        if input_t_per_year == 0:
            return "not applicable (zero input)"
        return f"{outflow_t_per_year / input_t_per_year * 100:.1f}%"

    print("\n" + "=" * 50)
    print("Starting model diagnostics")
    print("=" * 50)

    print("=== 1. Aggregate flow analysis ===")
    recycled_material_codes_to_check = [
        "WC",
        "WB",
        "RCS",
        "RCL",
        "RCC",
        "RCF",
        "RBC",
        "RBF",
        "RBP",
    ]
    print("Total flow by material (t yr^-1):")
    for material_code in recycled_material_codes_to_check:
        material_flow_total_t_per_year = flow_variable_index_df.loc[
            (flow_variable_index_df["material_code"] == material_code)
            & (
                flow_variable_index_df["flow_t_per_year"]
                > REPORTING_THRESHOLD_T_PER_YEAR
            ),
            "flow_t_per_year",
        ].sum()
        print(f"  {material_code}: {material_flow_total_t_per_year:.6f}")

    print("\n=== 2. Material-flow chain checks ===")
    print(
        "Expected branches: NDGRD(WC/WB) -> NDWRP; RCS/RCL: NDWRP -> NDPRJ; "
        "RCC: NDWRP -> NDCBP -> NDPRJ; RCF/RBC/RBF/RBP: "
        "NDWRP -> NDBMP -> NDPRJ"
    )

    print("\n2.1 NDGRD-to-NDWRP waste transport:")
    waste_concrete_grid_to_recycling_df = flow_variable_index_df.loc[
        (flow_variable_index_df["origin_node_type"] == "NDGRD")
        & (flow_variable_index_df["destination_node_type"] == "NDWRP")
        & (flow_variable_index_df["material_code"] == "WC")
    ]
    waste_brick_grid_to_recycling_df = flow_variable_index_df.loc[
        (flow_variable_index_df["origin_node_type"] == "NDGRD")
        & (flow_variable_index_df["destination_node_type"] == "NDWRP")
        & (flow_variable_index_df["material_code"] == "WB")
    ]
    print(
        "  NDGRD-to-NDWRP WC arc count: "
        f"{len(waste_concrete_grid_to_recycling_df)}"
    )
    if len(waste_concrete_grid_to_recycling_df) > 0:
        positive_waste_concrete_grid_flows_df = (
            waste_concrete_grid_to_recycling_df.loc[
                waste_concrete_grid_to_recycling_df["flow_t_per_year"]
                > REPORTING_THRESHOLD_T_PER_YEAR
            ]
        )
        print(
            "    positive-flow arcs: "
            f"{len(positive_waste_concrete_grid_flows_df)}, total flow: "
            f"{waste_concrete_grid_to_recycling_df['flow_t_per_year'].sum():.6f} "
            "t yr^-1"
        )
    print(
        "  NDGRD-to-NDWRP WB arc count: "
        f"{len(waste_brick_grid_to_recycling_df)}"
    )
    if len(waste_brick_grid_to_recycling_df) > 0:
        positive_waste_brick_grid_flows_df = waste_brick_grid_to_recycling_df.loc[
            waste_brick_grid_to_recycling_df["flow_t_per_year"]
            > REPORTING_THRESHOLD_T_PER_YEAR
        ]
        print(
            "    positive-flow arcs: "
            f"{len(positive_waste_brick_grid_flows_df)}, total flow: "
            f"{waste_brick_grid_to_recycling_df['flow_t_per_year'].sum():.6f} "
            "t yr^-1"
        )

    print("\n2.2 Material flows from NDWRP to downstream nodes:")
    for material_code in ["RCS", "RCL", "RCC", "RCF", "RBC", "RBF", "RBP"]:
        recycling_plant_outflow_df = flow_variable_index_df.loc[
            (flow_variable_index_df["origin_node_type"] == "NDWRP")
            & (flow_variable_index_df["material_code"] == material_code)
            & (
                flow_variable_index_df["flow_t_per_year"]
                > REPORTING_THRESHOLD_T_PER_YEAR
            )
        ]
        if len(recycling_plant_outflow_df) > 0:
            total_recycling_plant_outflow_t_per_year = recycling_plant_outflow_df[
                "flow_t_per_year"
            ].sum()
            print(
                f"    NDWRP->* {material_code}: "
                f"{len(recycling_plant_outflow_df)} arcs, total flow: "
                f"{total_recycling_plant_outflow_t_per_year:.6f} t yr^-1"
            )

    print("\n2.3 RCF arc check:")
    rcf_recycling_to_block_df = flow_variable_index_df.loc[
        (flow_variable_index_df["origin_node_type"] == "NDWRP")
        & (flow_variable_index_df["destination_node_type"] == "NDBMP")
        & (flow_variable_index_df["material_code"] == "RCF")
    ]
    rcf_block_to_project_df = flow_variable_index_df.loc[
        (flow_variable_index_df["origin_node_type"] == "NDBMP")
        & (flow_variable_index_df["destination_node_type"] == "NDPRJ")
        & (flow_variable_index_df["material_code"] == "RCF")
    ]
    rcf_recycling_to_block_total_t_per_year = rcf_recycling_to_block_df.loc[
        rcf_recycling_to_block_df["flow_t_per_year"]
        > REPORTING_THRESHOLD_T_PER_YEAR,
        "flow_t_per_year",
    ].sum()
    rcf_block_to_project_total_t_per_year = rcf_block_to_project_df.loc[
        rcf_block_to_project_df["flow_t_per_year"]
        > REPORTING_THRESHOLD_T_PER_YEAR,
        "flow_t_per_year",
    ].sum()
    print(
        "    NDWRP->NDBMP RCF arc count: "
        f"{len(rcf_recycling_to_block_df)}; positive-flow total: "
        f"{rcf_recycling_to_block_total_t_per_year:.6f} t yr^-1"
    )
    print(
        "    NDBMP->NDPRJ RCF arc count: "
        f"{len(rcf_block_to_project_df)}; positive-flow total: "
        f"{rcf_block_to_project_total_t_per_year:.6f} t yr^-1"
    )

    print("\n=== 3. Recycling-plant conversion checks ===")
    if (
        scheme_throughput_index_df is not None
        and "throughput_t_per_year" in scheme_throughput_index_df.columns
    ):
        diagnostic_active_schemes_df = scheme_throughput_index_df.loc[
            scheme_throughput_index_df["throughput_t_per_year"]
            > REPORTING_THRESHOLD_T_PER_YEAR
        ]
        print(f"Active recycling-plant schemes: {len(diagnostic_active_schemes_df)}")
        for recycling_plant_id in recycling_plant_ids[:3]:
            plant_schemes_df = diagnostic_active_schemes_df.loc[
                diagnostic_active_schemes_df["recycling_plant_id"]
                == recycling_plant_id
            ]
            if len(plant_schemes_df) == 0:
                continue
            print(f"\n  {recycling_plant_id} recycling schemes:")
            for _, scheme_record in plant_schemes_df.iterrows():
                print(
                    f"    {scheme_record['recycling_scheme']}: "
                    f"{scheme_record['throughput_t_per_year']:.6f} t yr^-1"
                )

            waste_concrete_input_t_per_year = flow_variable_index_df.loc[
                (flow_variable_index_df["destination_node_id"] == recycling_plant_id)
                & (flow_variable_index_df["material_code"] == "WC")
                & (
                    flow_variable_index_df["flow_t_per_year"]
                    > REPORTING_THRESHOLD_T_PER_YEAR
                ),
                "flow_t_per_year",
            ].sum()
            waste_brick_input_t_per_year = flow_variable_index_df.loc[
                (flow_variable_index_df["destination_node_id"] == recycling_plant_id)
                & (flow_variable_index_df["material_code"] == "WB")
                & (
                    flow_variable_index_df["flow_t_per_year"]
                    > REPORTING_THRESHOLD_T_PER_YEAR
                ),
                "flow_t_per_year",
            ].sum()
            concrete_scheme_throughput_t_per_year = plant_schemes_df.loc[
                plant_schemes_df["recycling_scheme"].isin(
                    ["CRS-S", "CRS-L", "CRS-C"]
                ),
                "throughput_t_per_year",
            ].sum()
            brick_scheme_throughput_t_per_year = plant_schemes_df.loc[
                plant_schemes_df["recycling_scheme"].isin(
                    ["BRS-C", "BRS-F", "BRS-P"]
                ),
                "throughput_t_per_year",
            ].sum()
            print("    Feedstock-balance check (t yr^-1):")
            print(
                f"      WC input: {waste_concrete_input_t_per_year:.6f}, "
                "WC-scheme throughput: "
                f"{concrete_scheme_throughput_t_per_year:.6f}, difference: "
                f"{abs(waste_concrete_input_t_per_year - concrete_scheme_throughput_t_per_year):.6f}"
            )
            print(
                f"      WB input: {waste_brick_input_t_per_year:.6f}, "
                "WB-scheme throughput: "
                f"{brick_scheme_throughput_t_per_year:.6f}, difference: "
                f"{abs(waste_brick_input_t_per_year - brick_scheme_throughput_t_per_year):.6f}"
            )

    print("\n=== 4. Waste-feedstock input utilization checks ===")
    for grid_cell_id in demolition_waste_grid_cell_ids[:5]:
        waste_concrete_input_t_per_year = float(
            waste_concrete_input_by_grid_cell.at[grid_cell_id]
        )
        waste_brick_input_t_per_year = float(
            waste_brick_input_by_grid_cell.at[grid_cell_id]
        )
        waste_concrete_outflow_t_per_year = flow_variable_index_df.loc[
            (flow_variable_index_df["origin_node_id"] == grid_cell_id)
            & (flow_variable_index_df["material_code"] == "WC")
            & (
                flow_variable_index_df["flow_t_per_year"]
                > REPORTING_THRESHOLD_T_PER_YEAR
            ),
            "flow_t_per_year",
        ].sum()
        waste_brick_outflow_t_per_year = flow_variable_index_df.loc[
            (flow_variable_index_df["origin_node_id"] == grid_cell_id)
            & (flow_variable_index_df["material_code"] == "WB")
            & (
                flow_variable_index_df["flow_t_per_year"]
                > REPORTING_THRESHOLD_T_PER_YEAR
            ),
            "flow_t_per_year",
        ].sum()
        print(f"  {grid_cell_id}:")
        print(
            f"    WC network input: {waste_concrete_input_t_per_year:.2f} "
            f"t yr^-1, WC outflow: {waste_concrete_outflow_t_per_year:.6f} "
            "t yr^-1, utilization: "
            f"{utilization_text(waste_concrete_outflow_t_per_year, waste_concrete_input_t_per_year)}"
        )
        print(
            f"    WB network input: {waste_brick_input_t_per_year:.2f} "
            f"t yr^-1, WB outflow: {waste_brick_outflow_t_per_year:.6f} "
            "t yr^-1, utilization: "
            f"{utilization_text(waste_brick_outflow_t_per_year, waste_brick_input_t_per_year)}"
        )

    print("\n=== 5. Construction-project demand checks ===")
    project_delivery_flows_df = flow_variable_index_df.loc[
        (flow_variable_index_df["destination_node_type"] == "NDPRJ")
        & (
            flow_variable_index_df["flow_t_per_year"]
            > REPORTING_THRESHOLD_T_PER_YEAR
        )
    ]
    if len(project_delivery_flows_df) > 0:
        project_delivery_by_material = (
            project_delivery_flows_df.groupby("material_code")["flow_t_per_year"]
            .sum()
            .sort_values(ascending=False)
        )
        print("Material delivered to construction projects (t yr^-1):")
        for material_code, flow_t_per_year in project_delivery_by_material.items():
            print(f"  {material_code}: {flow_t_per_year:.6f}")
        total_project_flow_t_per_year = project_delivery_by_material.sum()
        total_demand_t_per_year = sum(
            require_node_parameter(project_id, "concrete_demand")
            + require_node_parameter(project_id, "block_demand")
            for project_id in construction_project_ids
        )
        print(
            "\nTotal flow to construction projects: "
            f"{total_project_flow_t_per_year:.6f} t yr^-1"
        )
        print(
            "Total construction-project demand: "
            f"{total_demand_t_per_year:.6f} t yr^-1"
        )
        if total_demand_t_per_year == 0:
            print("Demand satisfaction: not applicable (zero demand)")
        else:
            print(
                "Demand satisfaction: "
                f"{total_project_flow_t_per_year / total_demand_t_per_year * 100:.1f}%"
            )
    else:
        print("No material reaches construction projects; inspect inputs and constraints.")

    print("\n=== 6. Carbon-impact analysis ===")
    reported_flows_df = flow_variable_index_df.loc[
        flow_variable_index_df["flow_t_per_year"]
        > REPORTING_THRESHOLD_T_PER_YEAR
    ]
    if len(reported_flows_df) > 0:
        reported_flow_carbon_impact_kgco2e_per_year = (
            reported_flows_df["flow_t_per_year"]
            * reported_flows_df["unit_carbon_impact_kgco2e_per_t"]
        ).sum()
        print(
            "Carbon impact represented by flows above the reporting threshold: "
            f"{reported_flow_carbon_impact_kgco2e_per_year:.2f} "
            "kg CO2-eq yr^-1"
        )
        reported_carbon_impact_by_material = (
            reported_flows_df.groupby("material_code")
            .apply(
                lambda material_flows_df: (
                    material_flows_df["flow_t_per_year"]
                    * material_flows_df["unit_carbon_impact_kgco2e_per_t"]
                ).sum()
            )
            .sort_values(ascending=False)
        )
        print("Highest material-level carbon impacts (kg CO2-eq yr^-1):")
        for material_code, carbon_impact_kgco2e_per_year in (
            reported_carbon_impact_by_material.head(10).items()
        ):
            print(f"  {material_code}: {carbon_impact_kgco2e_per_year:.2f}")

    print("\nDiagnostics complete.")
    print("=" * 50)


if model.Status != GRB.OPTIMAL:
    print(
        "Diagnostics and result export skipped because no successful run "
        "is available. Existing files in the scenario results directory, "
        "if any, were not updated by this run."
    )
else:
    try:
        log_status("Extracting result values")
        flow_variable_index_df["flow_t_per_year"] = material_flow_vars.X

        if scheme_throughput_vars is not None and scheme_throughput_index_df is not None:
            scheme_throughput_index_df[
                "throughput_t_per_year"
            ] = scheme_throughput_vars.X
        log_status("Result extraction complete")

        report_solution_diagnostics()

        positive_flow_results_df = flow_variable_index_df.loc[
            flow_variable_index_df["flow_t_per_year"]
            > REPORTING_THRESHOLD_T_PER_YEAR
        ].copy()
        if "row_id" in positive_flow_results_df.columns:
            positive_flow_results_df.drop(columns=["row_id"], inplace=True)
        flow_output_columns = [
            "material_code",
            "origin_node_id",
            "destination_node_id",
            "distance_km",
            "arc_type",
            "origin_node_type",
            "destination_node_type",
            "unit_carbon_impact_kgco2e_per_t",
            "flow_t_per_year",
        ]
        positive_flow_results_df = positive_flow_results_df[flow_output_columns]
        positive_flow_results_df.to_csv(
            FLOW_RESULTS_PATH, index=False, encoding="utf-8-sig"
        )

        scheme_output_columns = [
            "recycling_plant_id",
            "recycling_scheme",
            "throughput_t_per_year",
        ]
        if (
            scheme_throughput_index_df is not None
            and "throughput_t_per_year" in scheme_throughput_index_df.columns
        ):
            active_schemes_df = scheme_throughput_index_df.loc[
                scheme_throughput_index_df["throughput_t_per_year"]
                > REPORTING_THRESHOLD_T_PER_YEAR,
                scheme_output_columns,
            ].copy()
        else:
            active_schemes_df = pd.DataFrame(columns=scheme_output_columns)
        active_schemes_df.to_csv(
            SCHEME_THROUGHPUT_RESULTS_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        print(f"\nFlow results saved: {len(positive_flow_results_df)} rows")
        print(f"Scheme-throughput results saved: {len(active_schemes_df)} rows")
        print("\nResult summary:")
        print(f"Positive-flow variable count: {len(positive_flow_results_df)}")
        print(
            "Material codes represented: "
            f"{positive_flow_results_df['material_code'].unique()}"
        )
        print(
            "Origin-node types represented: "
            f"{positive_flow_results_df['origin_node_type'].unique()}"
        )
        print(
            "Destination-node types represented: "
            f"{positive_flow_results_df['destination_node_type'].unique()}"
        )
        flow_by_material = (
            positive_flow_results_df.groupby("material_code")["flow_t_per_year"]
            .sum()
            .sort_values(ascending=False)
        )
        print("\nFlow by material (top 10; t yr^-1):")
        for material_code, flow_t_per_year in flow_by_material.head(10).items():
            print(f"  {material_code}: {flow_t_per_year:.2f}")
    except Exception as error:
        raise RuntimeError("Result extraction or export failed.") from error